# DS317 - Thuc nghiem M5: Chon Loc MI & Target log1p

- Doc lap noi tuyen `M5FeatureTarget`, khong can `m5_core`.
- Chay `B0, T1, S1, S2, ST1, ST2` x 7 models x 3 Fold.

In [ ]:
"""Cell 1: Cau hinh thuc nghiem, Cache & Co dieu khien."""

import os

SEED = 42
IS_KAGGLE = os.path.exists('/kaggle')
WORK_DIR = (
    '/kaggle/working/results_m5'
    if IS_KAGGLE
    else './local_results_m5'
)
os.makedirs(WORK_DIR, exist_ok=True)

MAX_TRIPS_PER_QUARTER = 2500
MAX_DL_WINDOWS_PER_SPLIT = 250_000
RECOVER_Q1_RAW_NANS = False
RESUME_FROM_DISK = True

ACTIVE_MODELS = [
    'LightGBM',
    'XGBoost',
    'CatBoost',
    'Bi-LSTM',
    'PatchTST',
    'TFT',
    'ST-GNN',
]

CFG_OVERRIDES = {
    'SEED': SEED,
    'MAX_TRIPS_PER_QUARTER': MAX_TRIPS_PER_QUARTER,
    'MAX_DL_WINDOWS_PER_SPLIT': MAX_DL_WINDOWS_PER_SPLIT,
    'RECOVER_Q1_RAW_NANS': RECOVER_Q1_RAW_NANS,
    'RESUME_FROM_DISK': RESUME_FROM_DISK,
    'ACTIVE_MODELS': ACTIVE_MODELS,
}
print(f'[CONFIG] WORK_DIR={WORK_DIR} | MODELS={len(ACTIVE_MODELS)}')

In [ ]:
"""DS317 Unified Experiment Engine v2 (Zero Leakage & Horizon-Stratified).

Loi Engine v2 dung chung cho Baseline (00) va 7 Module (M1..M7):
- Giu Eval tho (is_train=False) + bao cao 3 tang All / LE_168h / GT_168h.
- Va STGNNRegressor (LayerNorm + Residual) & _predict_lazy (perm xac dinh).
- Tach InnerVal thanh 2 luong: clean (Early Stopping) va raw_cal (M7).
- Va init_score/base_score & matched early stopping cho M6 custom loss.
- Ho tro checkpointing theo (exp_name, variant) va dong goi ZIP.
"""

# 1. Standard library imports
import gc
import glob
import os
import time
from typing import Any, Callable, Dict, List, Optional, Tuple
import warnings
import zipfile

# 2. Third-party imports
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.feature_selection import mutual_info_regression

try:
    import lightgbm as lgb
except ImportError:
    lgb = None

try:
    import xgboost as xgb
except ImportError:
    xgb = None

try:
    from catboost import CatBoostRegressor
except ImportError:
    CatBoostRegressor = None

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset
except ImportError:
    torch = None
    nn = None
    Dataset = object

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ======================================================================
# CAU HINH MAC DINH TOAN CUC
# ======================================================================
SEED = 42
IS_KAGGLE = os.path.exists('/kaggle')
HAS_CUDA = bool(torch is not None and torch.cuda.is_available())
MAX_TRIPS_PER_QUARTER: Optional[int] = 2500
MAX_DL_WINDOWS_PER_SPLIT: Optional[int] = 250_000
RECOVER_Q1_RAW_NANS: bool = False
RESUME_FROM_DISK: bool = True

ACTIVE_MODELS: List[str] = [
    'LightGBM',
    'XGBoost',
    'CatBoost',
    'Bi-LSTM',
    'PatchTST',
    'TFT',
    'ST-GNN',
]

EXPANDING_FOLDS: List[Dict[str, Any]] = [
    {'fold': 'Fold_1', 'train_quarters': ['Q1'], 'eval_quarter': 'Q2'},
    {'fold': 'Fold_2', 'train_quarters': ['Q1', 'Q2'], 'eval_quarter': 'Q3'},
    {
        'fold': 'Fold_3',
        'train_quarters': ['Q1', 'Q2', 'Q3'],
        'eval_quarter': 'Q4',
    },
]

FORBIDDEN_LEAKAGE_COLS: List[str] = [
    'mmsi',
    'trip_id',
    'base_date_time',
    'ata_time',
    'Target_Hours',
]
RAW_ANGLE_COLS: List[str] = ['cog', 'heading']

BASELINE_20_FEATURES: List[str] = [
    'longitude',
    'latitude',
    'sog',
    'cog_sin',
    'cog_cos',
    'heading_sin',
    'heading_cos',
    'length',
    'width',
    'draft',
    'cargo',
    'Area_Proxy',
    'Volume_Proxy',
    'status',
    'vessel_type',
    'Is_Tanker',
    'berth_lon',
    'berth_lat',
    'Dist_To_Berth_NM',
    'imputation_flag',
]

WEATHER_12_FEATURES: List[str] = [
    'rtma_domain',
    'weather_missing_flag',
    'rtma_TMP_C',
    'rtma_DPT_C',
    'rtma_PRES_hpa',
    'rtma_VIS_nm',
    'rtma_WIND_knots',
    'rtma_GUST_knots',
    'rtma_wdir_sin',
    'rtma_wdir_cos',
    'rtma_headwind_knots',
    'rtma_crosswind_knots',
]
WEATHER_10_CONT_FEATURES: List[str] = WEATHER_12_FEATURES[2:]

DISCRETE_BASELINE_COLS = frozenset({
    'status',
    'vessel_type',
    'Is_Tanker',
    'imputation_flag',
    'rtma_domain',
    'weather_missing_flag',
})

STRATUM_METRIC_KEYS: List[str] = [
    'N',
    'Share_pct',
    'MAE_hours',
    'RMSE_hours',
    'MedAE_hours',
    'R2',
    'MBE_bias_hours',
    'Underprediction_pct',
    'Overprediction_pct',
    'Within_3h_pct',
    'AsymCost_w2_5',
    'PICP_90_pct',
    'MPIW_hours',
]

_WEATHER_SIDECAR_CACHE: Dict[str, Any] = {
    'paths': None,
    'trip_ids': set(),
    'lookup': None,
}


def assert_no_leakage_in_features(feature_cols: List[str]) -> None:
    """Kiem tra nghiem ngat khong co cot ID/Target/goc tho trong X."""
    forbidden_set = set(FORBIDDEN_LEAKAGE_COLS) | set(RAW_ANGLE_COLS)
    found = [c for c in feature_cols if c in forbidden_set]
    assert not found, f'PHAT HIEN COT CAM / RO RI TRONG X: {found}'


# ======================================================================
# BO NAP DU LIEU 4 QUY & TACH TRAIN / INNER-VAL
# ======================================================================


def resolve_quarter_version_dir(
    version_name: str = '03_ban_tien_xu_ly_27_thuoc_tinh',
    quarter_dir: Optional[str] = None,
) -> str:
    """Do tim thu muc chua q1..q4.parquet cua phien ban duoc chon."""
    override = quarter_dir or os.environ.get('AIS_QUARTER_DIR')
    if override and os.path.isdir(override):
        for cand in (os.path.join(override, version_name), override):
            if os.path.isfile(os.path.join(cand, 'q1.parquet')):
                return cand

    candidates = [
        os.path.join(
            '/kaggle/input/noaa-ais-2025-full-year-tin-x-l-nn-tng',
            version_name,
        ),
        os.path.join(
            '/kaggle/input/datasets/kienvo134/'
            'noaa-ais-2025-full-year-tin-x-l-nn-tng',
            version_name,
        ),
        os.path.join('/kaggle/working', version_name),
        os.path.join('./kaggle_work_fullyear', version_name),
        os.path.join('.', version_name),
    ]
    for cand in candidates:
        if os.path.isfile(os.path.join(cand, 'q1.parquet')):
            return cand

    if os.path.isdir('/kaggle/input'):
        hits = glob.glob(
            f'/kaggle/input/**/{version_name}/q1.parquet', recursive=True
        )
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        f'Khong tim thay thu muc 4 quy cho phien ban {version_name}.'
    )


def _restore_q1_original_nans(
    df_q1: pd.DataFrame,
    v_dir_27: str,
    sel_trips: Optional[set],
) -> Tuple[pd.DataFrame, str]:
    """Khoi phuc trang thai khuyet NaN goc cho Q1 phuc vu thuc nghiem M2."""
    if len(df_q1) == 0 or 'imputation_flag' not in df_q1.columns:
        return df_q1, 'skipped_empty_or_no_flag'

    df_out = df_q1.copy()
    imp_mask = df_out['imputation_flag'].to_numpy() > 0

    raw_q1_path = os.path.join(
        os.path.dirname(v_dir_27), '01_ban_tho_18_thuoc_tinh', 'q1.parquet'
    )
    if not os.path.isfile(raw_q1_path) and os.path.isdir('/kaggle/input'):
        hits = glob.glob(
            '/kaggle/input/**/01_ban_tho_18_thuoc_tinh/q1.parquet',
            recursive=True,
        )
        if hits:
            raw_q1_path = hits[0]

    mode_used = 'from_03_mask'
    if os.path.isfile(raw_q1_path):
        raw_cols = [
            'mmsi',
            'trip_id',
            'base_date_time',
            'length',
            'width',
            'draft',
            'cog',
            'heading',
            'status',
        ]
        pf_raw = pq.ParquetFile(raw_q1_path)
        raw_chunks = []
        for batch in pf_raw.iter_batches(
            batch_size=1_000_000, columns=raw_cols
        ):
            dfb = batch.to_pandas()
            if sel_trips is not None:
                dfb = dfb[dfb['trip_id'].isin(sel_trips)]
            if len(dfb) > 0:
                raw_chunks.append(dfb)
        if raw_chunks:
            df_r = pd.concat(raw_chunks, ignore_index=True)
            df_r['base_date_time'] = pd.to_datetime(df_r['base_date_time'])
            df_r = df_r.drop_duplicates(
                subset=['mmsi', 'trip_id', 'base_date_time']
            )
            merged = df_out[
                ['mmsi', 'trip_id', 'base_date_time']
            ].merge(
                df_r,
                on=['mmsi', 'trip_id', 'base_date_time'],
                how='left',
            )
            for c in ['length', 'width', 'draft', 'cog', 'heading', 'status']:
                df_out[c] = merged[c].to_numpy(dtype=np.float32)
            mode_used = 'from_01_raw_18'
    elif imp_mask.any():
        for c in ['length', 'width', 'draft']:
            df_out.loc[imp_mask, c] = np.nan

    df_out['Area_Proxy'] = (df_out['length'] * df_out['width']).astype(
        np.float32
    )
    df_out['Volume_Proxy'] = (
        df_out['Area_Proxy'] * df_out['draft']
    ).astype(np.float32)
    with np.errstate(invalid='ignore'):
        df_out['cog_sin'] = np.sin(np.radians(df_out['cog'])).astype(
            np.float32
        )
        df_out['cog_cos'] = np.cos(np.radians(df_out['cog'])).astype(
            np.float32
        )
        df_out['heading_sin'] = np.sin(np.radians(df_out['heading'])).astype(
            np.float32
        )
        df_out['heading_cos'] = np.cos(np.radians(df_out['heading'])).astype(
            np.float32
        )
    has_missing = (
        df_out['length'].isna()
        | (df_out['length'] <= 0.0)
        | df_out['width'].isna()
        | (df_out['width'] <= 0.0)
        | df_out['draft'].isna()
        | (df_out['draft'] <= 0.0)
        | (df_out['draft'] > 25.5)
    )
    df_out['imputation_flag'] = np.where(has_missing, -1, 0).astype(np.int32)
    return df_out, mode_used


def load_quarter_df(
    quarter_name: str,
    version_name: str = '03_ban_tien_xu_ly_27_thuoc_tinh',
    max_trips: Optional[int] = MAX_TRIPS_PER_QUARTER,
    seed: int = SEED,
    recover_q1_raw_nans: bool = RECOVER_Q1_RAW_NANS,
    quarter_dir: Optional[str] = None,
) -> pd.DataFrame:
    """Nap du lieu 1 quy tu q{1..4}.parquet, lay mau theo tron chuyen."""
    v_dir = resolve_quarter_version_dir(version_name, quarter_dir=quarter_dir)
    fpath = os.path.join(v_dir, f'{quarter_name.lower()}.parquet')

    sel_trips: Optional[set] = None
    if max_trips is None:
        df = pd.read_parquet(fpath)
    else:
        t_df = pq.read_table(fpath, columns=['trip_id']).to_pandas()
        uniq_trips = t_df['trip_id'].unique()
        del t_df

        if len(uniq_trips) > max_trips:
            rng = np.random.default_rng(seed)
            sel_trips = set(
                rng.choice(uniq_trips, size=max_trips, replace=False).tolist()
            )
        else:
            sel_trips = set(uniq_trips.tolist())

        pf = pq.ParquetFile(fpath)
        chunks = []
        for batch in pf.iter_batches(batch_size=1_000_000):
            dfb = batch.to_pandas()
            sub = dfb[dfb['trip_id'].isin(sel_trips)]
            if len(sub) > 0:
                chunks.append(sub)
        df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

    if len(df) > 0:
        df['base_date_time'] = pd.to_datetime(df['base_date_time'])
        df['ata_time'] = pd.to_datetime(df['ata_time'])
        if quarter_name.upper() == 'Q1' and recover_q1_raw_nans:
            df, r_mode = _restore_q1_original_nans(df, v_dir, sel_trips)
            print(f'  [Q1 NaN Recovery] mode={r_mode}')
        df = df.sort_values(['trip_id', 'base_date_time']).reset_index(
            drop=True
        )
    return df


def load_quarter_cache(
    quarters: Optional[List[str]] = None,
    version_name: str = '03_ban_tien_xu_ly_27_thuoc_tinh',
    max_trips: Optional[int] = MAX_TRIPS_PER_QUARTER,
    seed: int = SEED,
    recover_q1_raw_nans: bool = RECOVER_Q1_RAW_NANS,
    quarter_dir: Optional[str] = None,
) -> Dict[str, pd.DataFrame]:
    """Nap 4 quy mot lan duy nhat vao cache de chia se cho nhieu variant."""
    q_list = quarters if quarters is not None else ['Q1', 'Q2', 'Q3', 'Q4']
    cache: Dict[str, pd.DataFrame] = {}
    for q_code in q_list:
        cache[q_code] = load_quarter_df(
            quarter_name=q_code,
            version_name=version_name,
            max_trips=max_trips,
            seed=seed,
            recover_q1_raw_nans=recover_q1_raw_nans,
            quarter_dir=quarter_dir,
        )
        print(
            f'  [Cache {q_code}] {len(cache[q_code]):,} rows | '
            f'{cache[q_code]["trip_id"].nunique():,} trips'
        )
    return cache


def split_train_inner_val_by_trip(
    df_train: pd.DataFrame,
    val_trip_ratio: float = 0.15,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Tach Fit / Inner-Val theo thu tu thoi gian cua tung trip_id."""
    trip_times = (
        df_train.groupby('trip_id')['ata_time']
        .max()
        .sort_values()
        .reset_index()
    )
    n_val = max(1, int(len(trip_times) * val_trip_ratio))
    val_trips = set(trip_times.iloc[-n_val:]['trip_id'].tolist())

    is_val = df_train['trip_id'].isin(val_trips)
    df_fit = df_train.loc[~is_val].copy().reset_index(drop=True)
    df_val = df_train.loc[is_val].copy().reset_index(drop=True)

    overlap = len(set(df_fit['trip_id']) & set(df_val['trip_id']))
    assert overlap == 0, f'Loi trung trip_id giua Fit va Inner-Val: {overlap}'
    return df_fit, df_val


# ======================================================================
# MODULE 1: LOC NGOAI LAI (BASELINE, M1-A SPEEDCAP, M1-B CONSISTENCY)
# ======================================================================


class Module1OutlierFilter:
    """M1 (Huynh Chi Kien): Loc ngoai lai & Lam sach nhieu AIS."""

    def __init__(
        self,
        max_dist_tanker_nm: float = 1800.0,
        max_dist_cargo_nm: float = 2300.0,
        min_implied_speed_kn: float = 1.62,
    ) -> None:
        """Khoi tao nguong loc hang hai chuan."""
        self.max_dist_tanker_nm = max_dist_tanker_nm
        self.max_dist_cargo_nm = max_dist_cargo_nm
        self.min_implied_speed_kn = min_implied_speed_kn

    def _apply_common_filters(
        self, df: pd.DataFrame, is_train: bool
    ) -> pd.DataFrame:
        if len(df) == 0:
            return df.copy()
        sog = pd.to_numeric(df['sog'], errors='coerce')
        st = pd.to_numeric(df['status'], errors='coerce')
        tgt = pd.to_numeric(df['Target_Hours'], errors='coerce')
        is_tanker = df['Is_Tanker'].to_numpy() == 1
        max_d = np.where(
            is_tanker, self.max_dist_tanker_nm, self.max_dist_cargo_nm
        )
        # Eval chi giu nhan huu han > 0, khong loc implied_spd.
        mask = (
            sog.between(1.0, 40.0)
            & (~st.isin([1.0, 5.0]))
            & (df['Dist_To_Berth_NM'].to_numpy() <= max_d)
            & tgt.notna()
            & np.isfinite(tgt.to_numpy(dtype=np.float64))
            & (tgt.to_numpy(dtype=np.float64) > 0.0)
        )
        out = df.loc[mask].copy()
        if is_train and len(out) > 0:
            tgt_tr = pd.to_numeric(out['Target_Hours'], errors='coerce')
            spd = np.where(
                tgt_tr > 0.0,
                out['Dist_To_Berth_NM'].to_numpy() / tgt_tr.to_numpy(),
                -1.0,
            )
            lbl_ok = (tgt_tr > 0.0) & (spd >= self.min_implied_speed_kn)
            out = out.loc[lbl_ok].copy()
        return out.reset_index(drop=True)

    def fit_transform_train(self, df_train: pd.DataFrame) -> pd.DataFrame:
        """Loc nhieu va loc toc do ngu ngam tren tap Fit / Val-clean."""
        return self._apply_common_filters(df_train, is_train=True)

    def transform_eval(self, df_eval: pd.DataFrame) -> pd.DataFrame:
        """Loc cam bien dau vao tren Eval / Val-cal (khong loc implied_spd)."""
        return self._apply_common_filters(df_eval, is_train=False)


def filter_ais_jumps(
    df: pd.DataFrame,
    use_sog_check: bool = False,
    max_gap_minutes: float = 60.0,
) -> pd.DataFrame:
    """Loc buoc nhay toa do GPS qua toc do tu ban tin truoc cung trip."""
    if df.empty:
        return df.copy()
    cols = [
        'mmsi',
        'trip_id',
        'base_date_time',
        'latitude',
        'longitude',
        'sog',
    ]
    pts = df[cols].reset_index(drop=True)
    for col in ['latitude', 'longitude', 'sog']:
        pts[col] = pd.to_numeric(pts[col], errors='coerce')
    pts['_time'] = pd.to_datetime(
        pts['base_date_time'], utc=True, errors='coerce'
    )
    usable = (
        pts['_time'].notna()
        & pts['mmsi'].notna()
        & pts['trip_id'].notna()
        & pts['latitude'].between(-90.0, 90.0)
        & pts['longitude'].between(-180.0, 180.0)
    )
    pts = pts.loc[usable].sort_values('_time', kind='stable')
    prev = pts.groupby(['mmsi', 'trip_id'], sort=False)[
        ['_time', 'latitude', 'longitude', 'sog']
    ].shift()

    gap_h = (pts['_time'] - prev['_time']).dt.total_seconds() / 3600.0
    valid_gap = gap_h.gt(0.0) & gap_h.le(max_gap_minutes / 60.0)
    lat_n = np.deg2rad(pts['latitude'].astype('float64'))
    lat_p = np.deg2rad(prev['latitude'].astype('float64'))
    dlon = np.deg2rad((pts['longitude'] - prev['longitude']).astype('float64'))
    a_val = (
        np.sin((lat_n - lat_p) / 2.0) ** 2
        + np.cos(lat_n) * np.cos(lat_p) * np.sin(dlon / 2.0) ** 2
    )
    dist_nm = 2.0 * 3440.065 * np.arcsin(np.sqrt(a_val.clip(0.0, 1.0)))
    implied_kn = dist_nm / gap_h.where(valid_gap)

    reject = implied_kn.gt(45.0)
    if use_sog_check:
        max_sog = np.maximum(pts['sog'], prev['sog'])
        reject = reject | (
            implied_kn.gt(30.0) & (implied_kn - max_sog).gt(20.0)
        )

    keep = np.ones(len(df), dtype=bool)
    keep[pts.index[reject.fillna(False)]] = False
    return df.iloc[keep].reset_index(drop=True)


class Module1SpeedCap(Module1OutlierFilter):
    """M1-A: Loc nhay toa do > 45 hai ly/gio trong cua so <= 60 phut."""

    use_sog_check: bool = False

    def _apply_common_filters(
        self, df: pd.DataFrame, is_train: bool
    ) -> pd.DataFrame:
        screened = filter_ais_jumps(df, use_sog_check=self.use_sog_check)
        return super()._apply_common_filters(screened, is_train=is_train)


class Module1SpeedConsistency(Module1SpeedCap):
    """M1-B: Loc nhay toa do > 45 kn hoac (> 30 kn & lech max_sog > 20)."""

    use_sog_check: bool = True


# ======================================================================
# MODULE 2: DIEN KHUYET 3 CAP & CAC BIEN THE V0..V4
# ======================================================================


class Module2Imputer:
    """M2 (Do Van Loc): Dien khuyet 3 cap + V1 goc, V2 noi suy, V3 hinh hoc."""

    def __init__(
        self,
        use_cross_angle: bool = False,
        use_traj_interp: bool = False,
        use_hull_geometry: bool = False,
        interp_limit: int = 6,
    ) -> None:
        """Khoi tao cau hinh dien khuyet V0..V4."""
        self.use_cross_angle = bool(use_cross_angle)
        self.use_traj_interp = bool(use_traj_interp)
        self.use_hull_geometry = bool(use_hull_geometry)
        self.interp_limit = int(interp_limit)
        self.status_mode: float = 0.0
        self.global_medians: Dict[str, float] = {}
        self.group_medians: Dict[str, Dict[str, float]] = {}
        self.type_medians: Dict[str, Dict[int, float]] = {}
        self.mmsi_medians: Dict[str, Dict[int, float]] = {}
        self.last_transform_stats: Dict[str, int] = {}

    def fit(self, df_train: pd.DataFrame) -> 'Module2Imputer':
        """Hoc trung vi 3 cap va status_mode CHI tu du lieu goc (flag==0)."""
        if 'imputation_flag' in df_train.columns:
            orig_mask = df_train['imputation_flag'].to_numpy() == 0
            df_src = (
                df_train.loc[orig_mask] if orig_mask.any() else df_train
            )
        else:
            df_src = df_train

        st_v = pd.to_numeric(df_src['status'], errors='coerce').dropna()
        self.status_mode = (
            float(st_v.mode().iloc[0]) if len(st_v) > 0 else 0.0
        )

        is_tanker = df_src['Is_Tanker'].to_numpy() == 1
        for col in ['length', 'width', 'draft']:
            vals = pd.to_numeric(df_src[col], errors='coerce')
            if col == 'draft':
                ok = vals.notna() & (vals > 0.0) & (vals <= 25.5)
            else:
                ok = vals.notna() & (vals > 0.0)

            sub = df_src.loc[ok, ['mmsi', 'vessel_type']].copy()
            sub['val'] = vals.loc[ok].astype(np.float64)
            sub['is_tanker'] = is_tanker[ok.to_numpy()]

            g_med = float(sub['val'].median()) if len(sub) > 0 else 0.0
            self.global_medians[col] = g_med

            t_sub = sub.loc[sub['is_tanker'], 'val']
            c_sub = sub.loc[~sub['is_tanker'], 'val']
            self.group_medians[col] = {
                'tanker': float(t_sub.median()) if len(t_sub) > 0 else g_med,
                'cargo': float(c_sub.median()) if len(c_sub) > 0 else g_med,
            }
            self.type_medians[col] = (
                sub.groupby('vessel_type')['val'].median().to_dict()
                if len(sub) > 0
                else {}
            )
            self.mmsi_medians[col] = (
                sub.groupby('mmsi')['val'].median().to_dict()
                if len(sub) > 0
                else {}
            )
        return self

    @staticmethod
    def _valid_angle_mask(arr: np.ndarray) -> np.ndarray:
        """Goc hop le AIS nam trong [0, 360) va khong phai 511."""
        return np.isfinite(arr) & (arr >= 0.0) & (arr < 360.0)

    def _apply_traj_interp(
        self,
        out: pd.DataFrame,
        cog_sin: np.ndarray,
        cog_cos: np.ndarray,
        hdg_sin: np.ndarray,
        hdg_cos: np.ndarray,
        cog_ok: np.ndarray,
        hdg_ok: np.ndarray,
    ) -> Tuple[
        np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, int
    ]:
        """Noi suy tuyen tinh gioi han quan tinh trong tung trip_id."""
        n_rows = len(out)
        sog_arr = pd.to_numeric(out['sog'], errors='coerce').to_numpy(
            dtype=np.float64, copy=True
        )
        bdt_ns = (
            pd.to_datetime(out['base_date_time'])
            .to_numpy(dtype='datetime64[ns]')
            .view(np.int64)
        )
        trip_codes = pd.factorize(out['trip_id'], sort=False)[0]
        order = np.lexsort((bdt_ns, trip_codes))
        inv = np.empty(n_rows, dtype=np.int64)
        inv[order] = np.arange(n_rows, dtype=np.int64)

        df_tmp = pd.DataFrame({
            'trip': trip_codes[order],
            'sog': sog_arr[order],
            'cs': np.where(cog_ok[order], cog_sin[order], np.nan),
            'cc': np.where(cog_ok[order], cog_cos[order], np.nan),
            'hs': np.where(hdg_ok[order], hdg_sin[order], np.nan),
            'hc': np.where(hdg_ok[order], hdg_cos[order], np.nan),
        })
        n_interp = 0
        for col in ['sog', 'cs', 'cc', 'hs', 'hc']:
            na_before = int(df_tmp[col].isna().sum())
            if na_before > 0:
                df_tmp[col] = df_tmp.groupby(
                    'trip', sort=False
                )[col].transform(
                    lambda s: s.interpolate(
                        method='linear',
                        limit=self.interp_limit,
                        limit_area='inside',
                    )
                )
                na_after = int(df_tmp[col].isna().sum())
                n_interp += na_before - na_after

        sog_out = np.nan_to_num(
            df_tmp['sog'].to_numpy(dtype=np.float32)[inv], nan=0.0
        )
        cs_out = df_tmp['cs'].to_numpy(dtype=np.float32)[inv]
        cc_out = df_tmp['cc'].to_numpy(dtype=np.float32)[inv]
        hs_out = df_tmp['hs'].to_numpy(dtype=np.float32)[inv]
        hc_out = df_tmp['hc'].to_numpy(dtype=np.float32)[inv]

        for s_arr, c_arr in ((cs_out, cc_out), (hs_out, hc_out)):
            norm = np.hypot(s_arr, c_arr)
            good = np.isfinite(norm) & (norm > 1e-6)
            s_arr[good] = s_arr[good] / norm[good]
            c_arr[good] = c_arr[good] / norm[good]
            bad = ~good
            s_arr[bad] = 0.0
            c_arr[bad] = 1.0

        return (
            sog_out.astype(np.float32),
            cs_out.astype(np.float32),
            cc_out.astype(np.float32),
            hs_out.astype(np.float32),
            hc_out.astype(np.float32),
            n_interp,
        )

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Ap dung tham so da fit va cac cai tien M2 ma khong ro ri."""
        out = df.copy()
        n_rows = len(out)
        if n_rows == 0:
            self.last_transform_stats = {
                'hull_imputed': 0,
                'cross_angle_filled': 0,
                'traj_interpolated': 0,
                'geometry_repaired': 0,
            }
            return out

        mmsi_s = out['mmsi']
        vt_s = out['vessel_type']
        is_tanker = out['Is_Tanker'].to_numpy() == 1
        existing_flag = (
            out['imputation_flag'].to_numpy(dtype=np.int32)
            if 'imputation_flag' in out.columns
            else np.zeros(n_rows, dtype=np.int32)
        )
        flag_list = [np.where(existing_flag > 0, existing_flag, 0)]
        hull_imputed = 0

        for col in ['length', 'width', 'draft']:
            arr = pd.to_numeric(out[col], errors='coerce').to_numpy(
                dtype=np.float32, copy=True
            )
            if col == 'draft':
                bad = np.isnan(arr) | (arr <= 0.0) | (arr > 25.5)
            else:
                bad = np.isnan(arr) | (arr <= 0.0)

            f_col = np.zeros(n_rows, dtype=np.int32)
            if bad.any():
                hull_imputed += int(bad.sum())
                m_fill = pd.to_numeric(
                    mmsi_s.map(self.mmsi_medians[col]), errors='coerce'
                ).to_numpy(dtype=np.float64)
                use_m = bad & (~np.isnan(m_fill))
                arr[use_m] = m_fill[use_m].astype(np.float32)
                f_col[use_m] = 1

                rem = bad & (~use_m)
                if rem.any():
                    vt_fill = pd.to_numeric(
                        vt_s.map(self.type_medians[col]), errors='coerce'
                    ).to_numpy(dtype=np.float64)
                    use_vt = rem & (~np.isnan(vt_fill))
                    arr[use_vt] = vt_fill[use_vt].astype(np.float32)
                    f_col[use_vt] = 2

                    rem2 = rem & (~use_vt)
                    if rem2.any():
                        grp_fill = np.where(
                            is_tanker,
                            self.group_medians[col]['tanker'],
                            self.group_medians[col]['cargo'],
                        ).astype(np.float32)
                        arr[rem2] = grp_fill[rem2]
                        f_col[rem2] = 3

            out[col] = arr
            flag_list.append(f_col)

        out['imputation_flag'] = np.maximum.reduce(flag_list).astype(np.int32)

        geom_repaired = 0
        if self.use_hull_geometry:
            len_a = out['length'].to_numpy(dtype=np.float32, copy=True)
            wid_a = out['width'].to_numpy(dtype=np.float32, copy=True)
            drf_a = out['draft'].to_numpy(dtype=np.float32, copy=True)
            bad_w = (len_a > 0.0) & (wid_a > 0.35 * len_a)
            if bad_w.any():
                geom_repaired = int(bad_w.sum())
                wid_a[bad_w] = (0.16 * len_a[bad_w]).astype(np.float32)
                drf_a[bad_w] = (0.45 * wid_a[bad_w]).astype(np.float32)
                out['width'] = wid_a
                out['draft'] = drf_a

        cog_raw = pd.to_numeric(out['cog'], errors='coerce').to_numpy(
            dtype=np.float32, copy=True
        )
        hdg_raw = pd.to_numeric(out['heading'], errors='coerce').to_numpy(
            dtype=np.float32, copy=True
        )
        cog_ok = self._valid_angle_mask(cog_raw)
        hdg_ok = self._valid_angle_mask(hdg_raw)
        cross_filled = 0
        if self.use_cross_angle:
            fill_c = (~cog_ok) & hdg_ok
            fill_h = (~hdg_ok) & cog_ok
            cross_filled = int(fill_c.sum() + fill_h.sum())
            cog_raw[fill_c] = hdg_raw[fill_c]
            hdg_raw[fill_h] = cog_raw[fill_h]
            cog_ok = cog_ok | fill_c
            hdg_ok = hdg_ok | fill_h

        cog_v = np.where(cog_ok, cog_raw, np.float32(0.0)).astype(np.float32)
        hdg_v = np.where(hdg_ok, hdg_raw, np.float32(0.0)).astype(np.float32)
        cog_sin = np.sin(np.radians(cog_v)).astype(np.float32)
        cog_cos = np.cos(np.radians(cog_v)).astype(np.float32)
        hdg_sin = np.sin(np.radians(hdg_v)).astype(np.float32)
        hdg_cos = np.cos(np.radians(hdg_v)).astype(np.float32)

        traj_interp = 0
        if self.use_traj_interp and 'trip_id' in out.columns:
            sog_v, cog_sin, cog_cos, hdg_sin, hdg_cos, traj_interp = (
                self._apply_traj_interp(
                    out,
                    cog_sin,
                    cog_cos,
                    hdg_sin,
                    hdg_cos,
                    cog_ok,
                    hdg_ok,
                )
            )
            out['sog'] = sog_v

        out['cog'] = cog_v
        out['heading'] = hdg_v
        out['cog_sin'] = cog_sin
        out['cog_cos'] = cog_cos
        out['heading_sin'] = hdg_sin
        out['heading_cos'] = hdg_cos
        out['status'] = (
            pd.to_numeric(out['status'], errors='coerce')
            .fillna(np.float32(self.status_mode))
            .astype(np.float32)
        )
        out['Area_Proxy'] = (out['length'] * out['width']).astype(np.float32)
        out['Volume_Proxy'] = (out['Area_Proxy'] * out['draft']).astype(
            np.float32
        )
        self.last_transform_stats = {
            'hull_imputed': hull_imputed,
            'cross_angle_filled': cross_filled,
            'traj_interpolated': traj_interp,
            'geometry_repaired': geom_repaired,
        }
        return out


class Module2_V0_Baseline(Module2Imputer):
    """M2-V0: Dien khuyet 3 cap chuan Baseline."""

    def __init__(self) -> None:
        super().__init__(False, False, False)


class Module2_V1_CrossAngle(Module2Imputer):
    """M2-V1: Bu tru cheo goc cog <-> heading, bo gan 0 mac dinh."""

    def __init__(self) -> None:
        super().__init__(True, False, False)


class Module2_V2_TrajInterp(Module2Imputer):
    """M2-V2: Noi suy chuoi quan tinh quy dao theo trip_id."""

    def __init__(self) -> None:
        super().__init__(False, True, False)


class Module2_V3_HullGeometry(Module2Imputer):
    """M2-V3: Rang buoc ty le hinh hoc than vo W <= 0.35 * L."""

    def __init__(self) -> None:
        super().__init__(False, False, True)


class Module2_V4_FullCombined(Module2Imputer):
    """M2-V4: To hop toan dien V1 + V2 + V3."""

    def __init__(self) -> None:
        super().__init__(True, True, True)


# ======================================================================
# MODULE 3: TAO DAC TRUNG HANG HAI (G1..G5 + EXP6_ALL)
# ======================================================================


class Module3FeatureCreator:
    """M3 Baseline (Quach Tuan Kiet): Giu nguyen dac trung dau vao."""

    def fit(self, df_train: pd.DataFrame) -> 'Module3FeatureCreator':
        """Fit tham so (Baseline khong can fit)."""
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Tra ve ban sao khong them dac trung."""
        return df.copy()


class Module3AdvancedFeatureCreator(Module3FeatureCreator):
    """M3 Mo rong: Tao toi da 20 dac trung moi qua 5 nhom nhan qua."""

    def __init__(self, groups: Optional[List[int]] = None) -> None:
        """Khoi tao danh sach nhom dac trung can kich hoat (1..5)."""
        self.groups = list(groups) if groups is not None else []

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Tao cac dac trung moi theo nhom cau hinh, dam bao Zero Leakage."""
        if len(df) == 0 or not self.groups:
            return df.copy()

        out = df.copy()
        n_rows = len(out)

        if 1 in self.groups:
            lat_s = np.radians(
                pd.to_numeric(out['latitude'], errors='coerce').to_numpy(
                    dtype=np.float64
                )
            )
            lon_s = np.radians(
                pd.to_numeric(out['longitude'], errors='coerce').to_numpy(
                    dtype=np.float64
                )
            )
            lat_b = np.radians(
                pd.to_numeric(out['berth_lat'], errors='coerce').to_numpy(
                    dtype=np.float64
                )
            )
            lon_b = np.radians(
                pd.to_numeric(out['berth_lon'], errors='coerce').to_numpy(
                    dtype=np.float64
                )
            )
            dlon = lon_b - lon_s
            y_b = np.sin(dlon) * np.cos(lat_b)
            x_b = (
                np.cos(lat_s) * np.sin(lat_b)
                - np.sin(lat_s) * np.cos(lat_b) * np.cos(dlon)
            )
            b_rad = np.arctan2(y_b, x_b)
            cos_b = np.cos(b_rad)
            sin_b = np.sin(b_rad)

            out['bearing_to_berth'] = (
                (np.degrees(b_rad) + 360.0) % 360.0
            ).astype(np.float32)

            cs = pd.to_numeric(out['cog_sin'], errors='coerce').to_numpy(
                dtype=np.float64
            )
            cc = pd.to_numeric(out['cog_cos'], errors='coerce').to_numpy(
                dtype=np.float64
            )
            app_cos = np.clip(cc * cos_b + cs * sin_b, -1.0, 1.0)
            app_sin = np.clip(cs * cos_b - cc * sin_b, -1.0, 1.0)
            out['approach_cos'] = app_cos.astype(np.float32)
            out['approach_sin'] = app_sin.astype(np.float32)

            sog = np.maximum(
                0.0,
                pd.to_numeric(out['sog'], errors='coerce').to_numpy(
                    dtype=np.float64
                ),
            )
            out['v_closing'] = (sog * app_cos).astype(np.float32)
            out['v_cross'] = (sog * app_sin).astype(np.float32)
            out['delta_lon'] = (
                pd.to_numeric(out['berth_lon'], errors='coerce')
                - pd.to_numeric(out['longitude'], errors='coerce')
            ).astype(np.float32)
            out['delta_lat'] = (
                pd.to_numeric(out['berth_lat'], errors='coerce')
                - pd.to_numeric(out['latitude'], errors='coerce')
            ).astype(np.float32)

        if 2 in self.groups:
            cs = pd.to_numeric(out['cog_sin'], errors='coerce').to_numpy(
                dtype=np.float64
            )
            cc = pd.to_numeric(out['cog_cos'], errors='coerce').to_numpy(
                dtype=np.float64
            )
            hs = pd.to_numeric(out['heading_sin'], errors='coerce').to_numpy(
                dtype=np.float64
            )
            hc = pd.to_numeric(out['heading_cos'], errors='coerce').to_numpy(
                dtype=np.float64
            )
            out['drift_cos'] = np.clip(cc * hc + cs * hs, -1.0, 1.0).astype(
                np.float32
            )
            out['drift_sin'] = np.clip(cs * hc - cc * hs, -1.0, 1.0).astype(
                np.float32
            )
            len_a = np.maximum(
                1.0,
                pd.to_numeric(out['length'], errors='coerce').to_numpy(
                    dtype=np.float32
                ),
            )
            wid_a = np.maximum(
                1.0,
                pd.to_numeric(out['width'], errors='coerce').to_numpy(
                    dtype=np.float32
                ),
            )
            drf_a = np.maximum(
                0.1,
                pd.to_numeric(out['draft'], errors='coerce').to_numpy(
                    dtype=np.float32
                ),
            )
            out['wetted_surface_proxy'] = (
                len_a * (2.0 * drf_a + wid_a)
            ).astype(np.float32)
            out['slenderness_ratio'] = (
                len_a / np.sqrt(np.maximum(wid_a * drf_a, 0.1))
            ).astype(np.float32)
            out['block_draft_ratio'] = (
                drf_a / np.sqrt(np.maximum(len_a * wid_a, 1.0))
            ).astype(np.float32)

        if 3 in self.groups:
            dist = np.maximum(
                0.0,
                pd.to_numeric(
                    out['Dist_To_Berth_NM'], errors='coerce'
                ).to_numpy(dtype=np.float64),
            )
            sog = np.maximum(
                0.0,
                pd.to_numeric(out['sog'], errors='coerce').to_numpy(
                    dtype=np.float64
                ),
            )
            out['eta_kinematic_hours'] = np.clip(
                dist / np.maximum(sog, 0.5), 0.0, 1000.0
            ).astype(np.float32)
            if 'v_closing' in out.columns:
                v_c = out['v_closing'].to_numpy(dtype=np.float64)
            else:
                lat_s = np.radians(
                    pd.to_numeric(out['latitude'], errors='coerce').to_numpy(
                        dtype=np.float64
                    )
                )
                lon_s = np.radians(
                    pd.to_numeric(out['longitude'], errors='coerce').to_numpy(
                        dtype=np.float64
                    )
                )
                lat_b = np.radians(
                    pd.to_numeric(out['berth_lat'], errors='coerce').to_numpy(
                        dtype=np.float64
                    )
                )
                lon_b = np.radians(
                    pd.to_numeric(out['berth_lon'], errors='coerce').to_numpy(
                        dtype=np.float64
                    )
                )
                dlon = lon_b - lon_s
                br = np.arctan2(
                    np.sin(dlon) * np.cos(lat_b),
                    np.cos(lat_s) * np.sin(lat_b)
                    - np.sin(lat_s) * np.cos(lat_b) * np.cos(dlon),
                )
                cs = pd.to_numeric(out['cog_sin'], errors='coerce').to_numpy(
                    dtype=np.float64
                )
                cc = pd.to_numeric(out['cog_cos'], errors='coerce').to_numpy(
                    dtype=np.float64
                )
                v_c = sog * np.clip(
                    cc * np.cos(br) + cs * np.sin(br), -1.0, 1.0
                )
            out['eta_closing_hours'] = np.clip(
                dist / np.maximum(v_c, 0.5), 0.0, 1000.0
            ).astype(np.float32)
            out['log_dist_to_berth'] = np.log1p(dist).astype(np.float32)

        if (4 in self.groups or 5 in self.groups) and (
            'trip_id' in out.columns and 'base_date_time' in out.columns
        ):
            orig_order = np.arange(n_rows)
            sort_df = pd.DataFrame({
                'trip_id': out['trip_id'].values,
                'base_date_time': pd.to_datetime(out['base_date_time']).values,
                'orig_order': orig_order,
            }).sort_values(['trip_id', 'base_date_time'])
            idx_s = sort_df['orig_order'].to_numpy()
            trip_s = sort_df['trip_id'].values
            sog_s = (
                pd.to_numeric(out['sog'].iloc[idx_s], errors='coerce')
                .fillna(0.0)
                .to_numpy(dtype=np.float64)
            )
            t_sec = (
                pd.to_datetime(sort_df['base_date_time'])
                .to_numpy(dtype='datetime64[ns]')
                .view(np.int64)
                // 10**9
            )

            same_t = np.zeros(n_rows, dtype=bool)
            same_t[1:] = trip_s[:-1] == trip_s[1:]
            dt_s = np.zeros(n_rows, dtype=np.float64)
            dt_s[1:] = np.where(
                same_t[1:], np.maximum(0.0, t_sec[1:] - t_sec[:-1]), 0.0
            )
            dt_h = np.where(
                same_t, np.clip(dt_s / 3600.0, 1.0 / 60.0, 24.0), 1.0
            )

            if 4 in self.groups:
                d_sog = np.zeros(n_rows, dtype=np.float64)
                d_sog[1:] = np.where(same_t[1:], sog_s[1:] - sog_s[:-1], 0.0)
                acc = np.where(same_t, d_sog / dt_h, 0.0)

                cs_s = (
                    pd.to_numeric(out['cog_sin'].iloc[idx_s], errors='coerce')
                    .fillna(0.0)
                    .to_numpy(dtype=np.float64)
                )
                cc_s = (
                    pd.to_numeric(out['cog_cos'].iloc[idx_s], errors='coerce')
                    .fillna(0.0)
                    .to_numpy(dtype=np.float64)
                )
                d_sin = np.zeros(n_rows, dtype=np.float64)
                d_cos = np.ones(n_rows, dtype=np.float64)
                d_sin[1:] = np.where(
                    same_t[1:],
                    cs_s[1:] * cc_s[:-1] - cc_s[1:] * cs_s[:-1],
                    0.0,
                )
                d_cos[1:] = np.where(
                    same_t[1:],
                    cc_s[1:] * cc_s[:-1] + cs_s[1:] * cs_s[:-1],
                    1.0,
                )
                rot = np.where(
                    same_t,
                    np.degrees(np.arctan2(d_sin, d_cos)) / (dt_h * 60.0),
                    0.0,
                )

                dist_s = (
                    pd.to_numeric(
                        out['Dist_To_Berth_NM'].iloc[idx_s], errors='coerce'
                    )
                    .fillna(0.0)
                    .to_numpy(dtype=np.float64)
                )
                d_dist = np.zeros(n_rows, dtype=np.float64)
                d_dist[1:] = np.where(
                    same_t[1:], dist_s[:-1] - dist_s[1:], 0.0
                )
                v_dist = np.where(same_t, d_dist / dt_h, 0.0)
                drr = np.where(
                    same_t,
                    np.clip(v_dist / np.maximum(sog_s, 0.5), -5.0, 5.0),
                    1.0,
                )

                for col, vals in (
                    ('acc_sog', np.clip(acc, -20.0, 20.0)),
                    ('rot_approx', np.clip(rot, -60.0, 60.0)),
                    ('dist_rate_ratio', drr),
                ):
                    arr_out = np.zeros(n_rows, dtype=np.float32)
                    arr_out[idx_s] = vals.astype(np.float32)
                    out[col] = arr_out

            if 5 in self.groups:
                lag1 = np.copy(sog_s)
                lag2 = np.copy(sog_s)
                lag1[1:] = np.where(same_t[1:], sog_s[:-1], sog_s[1:])
                same_2 = np.zeros(n_rows, dtype=bool)
                same_2[2:] = same_t[1:-1] & same_t[2:]
                lag2[2:] = np.where(same_2[2:], sog_s[:-2], lag1[2:])

                roll_m = (sog_s + lag1 + lag2) / 3.0
                var_m = (
                    (sog_s - roll_m) ** 2
                    + (lag1 - roll_m) ** 2
                    + (lag2 - roll_m) ** 2
                ) / 3.0
                roll_std = np.sqrt(np.maximum(0.0, var_m))

                for col, vals in (
                    ('sog_rolling_mean_3', roll_m),
                    ('sog_rolling_std_3', roll_std),
                ):
                    arr_out = np.zeros(n_rows, dtype=np.float32)
                    arr_out[idx_s] = vals.astype(np.float32)
                    out[col] = arr_out

        return out


class Module3_Group1_BerthKinematics(Module3AdvancedFeatureCreator):
    """M3 Nhom 1: Dong hoc Hinh hoc Huong ben (+7 dac trung)."""

    def __init__(self) -> None:
        super().__init__(groups=[1])


class Module3_Group2_Hydrodynamics(Module3AdvancedFeatureCreator):
    """M3 Nhom 2: Thuy dong luc hoc & Luc can Than tau (+5 dac trung)."""

    def __init__(self) -> None:
        super().__init__(groups=[2])


class Module3_Group3_NonlinearETA(Module3AdvancedFeatureCreator):
    """M3 Nhom 3: Uoc luong Phi tuyen Thoi gian ETA (+3 dac trung)."""

    def __init__(self) -> None:
        super().__init__(groups=[3])


class Module3_Group4_Differentials(Module3AdvancedFeatureCreator):
    """M3 Nhom 4: Vi phan Chuyen dong Tuc thoi (+3 dac trung)."""

    def __init__(self) -> None:
        super().__init__(groups=[4])


class Module3_Group5_RollingStats(Module3AdvancedFeatureCreator):
    """M3 Nhom 5: Thong ke Cua so Qua khu (+2 dac trung)."""

    def __init__(self) -> None:
        super().__init__(groups=[5])


class Module3_AllCombined(Module3AdvancedFeatureCreator):
    """M3 Tong hop: Ca 5 nhom dac trung (+20 dac trung -> 40 cot)."""

    def __init__(self) -> None:
        super().__init__(groups=[1, 2, 3, 4, 5])


# ======================================================================
# MODULE 4: LAM GIAU THOI TIET NOAA RTMA 2025 (32 COT)
# ======================================================================


class Module4WeatherEnricher:
    """M4 Baseline (Vo Dinh Kien): Khong ghep RTMA."""

    def fit(self, df_train: pd.DataFrame) -> 'Module4WeatherEnricher':
        """Fit tham so (Baseline bo qua)."""
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Tra ve ban sao khong them dac trung thoi tiet."""
        return df.copy()


def resolve_weather_sidecar_paths(
    weather_dir: Optional[str] = None,
) -> List[str]:
    """Do tim duong dan train.parquet va test.parquet cua RTMA sidecar."""
    env_dir = weather_dir or os.environ.get('WEATHER_SIDECAR_DIR')
    if env_dir and os.path.isdir(env_dir):
        for cand_dir in (
            env_dir,
            os.path.join(env_dir, 'weather_rtma_2025_sidecar'),
        ):
            if os.path.isdir(cand_dir):
                hits = [
                    os.path.join(cand_dir, name)
                    for name in ('train.parquet', 'test.parquet')
                    if os.path.isfile(os.path.join(cand_dir, name))
                ]
                if hits:
                    return hits

    candidate_dirs = [
        '/kaggle/input/datasets/kienvo134/dataset-noaa-rtma-2025/'
        'weather_rtma_2025_sidecar',
        '/kaggle/input/dataset-noaa-rtma-2025/weather_rtma_2025_sidecar',
        '/kaggle/working/weather_rtma_2025_sidecar',
        './kaggle_work_rtma/weather_rtma_2025_sidecar',
    ]
    for cand_dir in candidate_dirs:
        hits = [
            os.path.join(cand_dir, name)
            for name in ('train.parquet', 'test.parquet')
            if os.path.isfile(os.path.join(cand_dir, name))
        ]
        if hits:
            return hits

    if os.path.isdir('/kaggle/input'):
        glob_hits = sorted(
            glob.glob(
                '/kaggle/input/**/weather_rtma_2025_sidecar/*.parquet',
                recursive=True,
            )
        )
        valid_hits = [
            p
            for p in glob_hits
            if os.path.basename(p) in ('train.parquet', 'test.parquet')
            and os.path.isfile(p)
        ]
        if valid_hits:
            return valid_hits

    raise FileNotFoundError(
        'Khong tim thay sidecar RTMA '
        '(weather_rtma_2025_sidecar/train.parquet hoac test.parquet).'
    )


def _normalize_mmsi(series: pd.Series) -> np.ndarray:
    return (
        pd.to_numeric(series, errors='coerce')
        .fillna(-1)
        .to_numpy(dtype=np.int64)
    )


def _normalize_bdt_us(series: pd.Series) -> np.ndarray:
    bdt = pd.to_datetime(series)
    if getattr(bdt.dt, 'tz', None) is not None:
        bdt = bdt.dt.tz_convert('UTC').dt.tz_localize(None)
    return bdt.to_numpy(dtype='datetime64[us]').view(np.int64)


def _empty_sidecar_lookup() -> pd.DataFrame:
    data: Dict[str, Any] = {
        '_k_mmsi': pd.Series(dtype=np.int64),
        '_k_trip': pd.Series(dtype=str),
        '_k_bdt': pd.Series(dtype=np.int64),
        '_sidecar_matched': pd.Series(dtype=np.int8),
        'rtma_domain': pd.Series(dtype=np.int8),
        'weather_missing_flag': pd.Series(dtype=np.int8),
    }
    for col in WEATHER_10_CONT_FEATURES:
        data[col] = pd.Series(dtype=np.float32)
    return pd.DataFrame(data)


def _load_sidecar_lookup_for_trips(
    target_trips: set,
    paths: Optional[List[str]] = None,
) -> pd.DataFrame:
    if not target_trips:
        return _empty_sidecar_lookup()

    sidecar_paths = (
        paths if paths is not None else resolve_weather_sidecar_paths()
    )
    read_cols = ['mmsi', 'trip_id', 'base_date_time'] + WEATHER_12_FEATURES
    chunks: List[pd.DataFrame] = []

    for fpath in sidecar_paths:
        pf = pq.ParquetFile(fpath)
        for batch in pf.iter_batches(batch_size=1_000_000, columns=read_cols):
            dfb = batch.to_pandas()
            trip_str = dfb['trip_id'].astype(str)
            mask = trip_str.isin(target_trips)
            if not mask.any():
                continue
            sub = dfb.loc[mask]
            chunk_data: Dict[str, Any] = {
                '_k_mmsi': _normalize_mmsi(sub['mmsi']),
                '_k_trip': trip_str.loc[mask].reset_index(drop=True),
                '_k_bdt': _normalize_bdt_us(sub['base_date_time']),
                '_sidecar_matched': np.ones(len(sub), dtype=np.int8),
                'rtma_domain': (
                    pd.to_numeric(sub['rtma_domain'], errors='coerce')
                    .fillna(0)
                    .to_numpy(dtype=np.int8)
                ),
                'weather_missing_flag': (
                    pd.to_numeric(
                        sub['weather_missing_flag'], errors='coerce'
                    )
                    .fillna(1)
                    .to_numpy(dtype=np.int8)
                ),
            }
            for col in WEATHER_10_CONT_FEATURES:
                chunk_data[col] = pd.to_numeric(
                    sub[col], errors='coerce'
                ).to_numpy(dtype=np.float32)
            chunks.append(pd.DataFrame(chunk_data))

    if not chunks:
        return _empty_sidecar_lookup()

    lookup = pd.concat(chunks, ignore_index=True)
    return lookup.drop_duplicates(
        subset=['_k_mmsi', '_k_trip', '_k_bdt'], keep='first'
    ).reset_index(drop=True)


def _get_cached_sidecar_lookup(
    target_trips: set, weather_dir: Optional[str] = None
) -> pd.DataFrame:
    paths = tuple(resolve_weather_sidecar_paths(weather_dir=weather_dir))
    if _WEATHER_SIDECAR_CACHE['paths'] != paths:
        _WEATHER_SIDECAR_CACHE['paths'] = paths
        _WEATHER_SIDECAR_CACHE['trip_ids'] = set()
        _WEATHER_SIDECAR_CACHE['lookup'] = None

    cached_trips: set = _WEATHER_SIDECAR_CACHE['trip_ids']
    missing_trips = target_trips - cached_trips
    if _WEATHER_SIDECAR_CACHE['lookup'] is None or missing_trips:
        to_scan = (
            missing_trips
            if _WEATHER_SIDECAR_CACHE['lookup'] is not None
            else target_trips
        )
        new_lk = _load_sidecar_lookup_for_trips(to_scan, paths=list(paths))
        if _WEATHER_SIDECAR_CACHE['lookup'] is None:
            combined = new_lk
        elif len(new_lk) > 0:
            combined = pd.concat(
                [_WEATHER_SIDECAR_CACHE['lookup'], new_lk], ignore_index=True
            ).drop_duplicates(
                subset=['_k_mmsi', '_k_trip', '_k_bdt'], keep='first'
            ).reset_index(drop=True)
        else:
            combined = _WEATHER_SIDECAR_CACHE['lookup']
        _WEATHER_SIDECAR_CACHE['lookup'] = combined
        _WEATHER_SIDECAR_CACHE['trip_ids'] = cached_trips | target_trips

    lookup_all = _WEATHER_SIDECAR_CACHE['lookup']
    if lookup_all is None or len(lookup_all) == 0 or not target_trips:
        return _empty_sidecar_lookup()
    return lookup_all[lookup_all['_k_trip'].isin(target_trips)]


class Module4RTMAWeatherEnricher(Module4WeatherEnricher):
    """M4 (Vo Dinh Kien): Ghep sidecar thoi tiet RTMA 2025 chong ro ri."""

    weather_dir: Optional[str] = None

    def __init__(self) -> None:
        """Khoi tao bo luu trung vi Train-Only cho 10 bien thoi tiet."""
        super().__init__()
        self.weather_medians: Dict[str, float] = {}

    @classmethod
    def warm_cache_from_quarters(
        cls,
        q_cache: Dict[str, pd.DataFrame],
        weather_dir: Optional[str] = None,
    ) -> None:
        """Nap truoc lookup sidecar mot lan cho toan bo trip trong 4 quy."""
        all_trips: set = set()
        for df_q in q_cache.values():
            if len(df_q) > 0 and 'trip_id' in df_q.columns:
                all_trips.update(df_q['trip_id'].astype(str).unique().tolist())

        use_dir = weather_dir or cls.weather_dir
        paths = tuple(resolve_weather_sidecar_paths(weather_dir=use_dir))
        lookup = _load_sidecar_lookup_for_trips(all_trips, paths=list(paths))
        _WEATHER_SIDECAR_CACHE['paths'] = paths
        _WEATHER_SIDECAR_CACHE['trip_ids'] = all_trips
        _WEATHER_SIDECAR_CACHE['lookup'] = lookup
        print(
            f'[M4 Cache] Nap sidecar 1 lan cho {len(all_trips):,} trips '
            f'-> {len(lookup):,} dong tra cuu.'
        )

    def _join_with_match_mask(
        self, df: pd.DataFrame
    ) -> Tuple[pd.DataFrame, np.ndarray]:
        out = df.copy()
        n_rows = len(out)
        if n_rows == 0:
            out['rtma_domain'] = np.empty(0, dtype=np.int8)
            out['weather_missing_flag'] = np.empty(0, dtype=np.int8)
            for col in WEATHER_10_CONT_FEATURES:
                out[col] = np.empty(0, dtype=np.float32)
            return out, np.empty(0, dtype=bool)

        target_trips = set(out['trip_id'].astype(str).unique().tolist())
        sub_lk = _get_cached_sidecar_lookup(
            target_trips, weather_dir=self.weather_dir
        )
        left_keys = pd.DataFrame({
            '_k_mmsi': _normalize_mmsi(out['mmsi']),
            '_k_trip': out['trip_id'].astype(str).reset_index(drop=True),
            '_k_bdt': _normalize_bdt_us(out['base_date_time']),
        })
        merged = left_keys.merge(
            sub_lk,
            on=['_k_mmsi', '_k_trip', '_k_bdt'],
            how='left',
            sort=False,
            validate='many_to_one',
        )
        assert len(merged) == n_rows

        matched = (
            pd.to_numeric(merged['_sidecar_matched'], errors='coerce')
            .fillna(0)
            .to_numpy(dtype=np.int8)
            == 1
        )
        out['rtma_domain'] = np.where(
            matched,
            pd.to_numeric(merged['rtma_domain'], errors='coerce')
            .fillna(0)
            .to_numpy(dtype=np.int8),
            np.int8(0),
        ).astype(np.int8)
        out['weather_missing_flag'] = np.where(
            matched,
            pd.to_numeric(merged['weather_missing_flag'], errors='coerce')
            .fillna(1)
            .to_numpy(dtype=np.int8),
            np.int8(1),
        ).astype(np.int8)
        for col in WEATHER_10_CONT_FEATURES:
            out[col] = pd.to_numeric(merged[col], errors='coerce').to_numpy(
                dtype=np.float32
            )
        return out, matched

    def fit(self, df_train: pd.DataFrame) -> 'Module4RTMAWeatherEnricher':
        """Hoc trung vi Train-Only tren cac gia tri huu han cua 10 bien."""
        if all(c in df_train.columns for c in WEATHER_12_FEATURES):
            df_j = df_train
        else:
            df_j, _ = self._join_with_match_mask(df_train)
        self.weather_medians = {}
        for col in WEATHER_10_CONT_FEATURES:
            arr = pd.to_numeric(df_j[col], errors='coerce').to_numpy(
                dtype=np.float64
            )
            fin = arr[np.isfinite(arr)]
            self.weather_medians[col] = (
                float(np.median(fin)) if len(fin) > 0 else 0.0
            )
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Ghep sidecar va dien khuyet bang trung vi Train-Only."""
        n_rows = len(df)
        if n_rows == 0:
            out, _ = self._join_with_match_mask(df)
            return out

        if all(c in df.columns for c in WEATHER_12_FEATURES):
            out = df.copy()
        else:
            out, _ = self._join_with_match_mask(df)

        out['rtma_domain'] = (
            pd.to_numeric(out['rtma_domain'], errors='coerce')
            .fillna(0)
            .to_numpy(dtype=np.int8)
        )
        out['weather_missing_flag'] = (
            pd.to_numeric(out['weather_missing_flag'], errors='coerce')
            .fillna(1)
            .to_numpy(dtype=np.int8)
        )
        for col in WEATHER_10_CONT_FEATURES:
            fill_val = np.float32(self.weather_medians.get(col, 0.0))
            arr = pd.to_numeric(out[col], errors='coerce').to_numpy(
                dtype=np.float32, copy=True
            )
            out[col] = np.where(np.isfinite(arr), arr, fill_val).astype(
                np.float32
            )
        return out


# ======================================================================
# MODULE 5: CHON LOC DAC TRUNG (AUTO + MI) & BIEN DOI TARGET (LOG1P)
# ======================================================================


class Module5FeatureTargetSelector:
    """M5 (Nguyen Huu Bao Loc): Tu dong giu toan bo dac trung sach."""

    def __init__(
        self,
        candidate_features: Optional[List[str]] = None,
    ) -> None:
        """Khoi tao danh sach ung vien (mac dinh None = auto non-leakage)."""
        self._explicit_candidates: Optional[List[str]] = (
            list(candidate_features)
            if candidate_features is not None
            else None
        )
        self.candidate_features: List[str] = (
            list(candidate_features)
            if candidate_features is not None
            else list(BASELINE_20_FEATURES)
        )
        self.selected_features: List[str] = []

    def fit(
        self, df_train: pd.DataFrame
    ) -> 'Module5FeatureTargetSelector':
        """Chon toan bo dac trung sach khong thuoc nhom cam/ro ri."""
        if self._explicit_candidates is not None:
            self.selected_features = [
                c for c in self._explicit_candidates if c in df_train.columns
            ]
        else:
            forbidden = set(FORBIDDEN_LEAKAGE_COLS) | set(RAW_ANGLE_COLS)
            base_cols = [
                c for c in BASELINE_20_FEATURES if c in df_train.columns
            ]
            extra_cols = [
                c
                for c in df_train.columns
                if c not in forbidden and c not in base_cols
            ]
            self.selected_features = base_cols + extra_cols
        assert_no_leakage_in_features(self.selected_features)
        return self

    def transform_x(self, df: pd.DataFrame) -> pd.DataFrame:
        """Trich xuat ma tran dac trung X chuan sach."""
        assert_no_leakage_in_features(self.selected_features)
        return df[self.selected_features].copy()

    def transform_y(self, y: pd.Series) -> np.ndarray:
        """Giu nguyen bien muc tieu goc Target_Hours."""
        return np.asarray(y, dtype=np.float32)

    def inverse_transform_y(self, y_pred: np.ndarray) -> np.ndarray:
        """Chuyen doi nguoc ve don vi gio (dong nhat)."""
        return np.asarray(y_pred, dtype=np.float32)


class M5FeatureTarget(Module5FeatureTargetSelector):
    """M5 Mo rong: Chon loc bang MI (giu cap sin/cos) va bien doi log1p."""

    ATOMIC_GROUPS: List[Tuple[str, ...]] = [
        ('cog_sin', 'cog_cos'),
        ('heading_sin', 'heading_cos'),
        ('rtma_wdir_sin', 'rtma_wdir_cos'),
    ]

    def __init__(
        self,
        k_features: Optional[int] = None,
        target_transform: str = 'raw',
        max_mi_rows: int = 100_000,
        seed: int = SEED,
        candidate_features: Optional[List[str]] = None,
    ) -> None:
        """Khoi tao cau hinh M5 (B0, T1, S1, S2, ST1, ST2)."""
        super().__init__(candidate_features=candidate_features)
        self.k_features = k_features
        self.target_transform = target_transform
        self.max_mi_rows = int(max_mi_rows)
        self.seed = int(seed)
        self.mi_scores: Dict[str, float] = {}

    def fit(self, df_train: pd.DataFrame) -> 'M5FeatureTarget':
        """Tinh Mutual Information tren Fit (<= 100k dong) va chon top k."""
        super().fit(df_train)
        all_feats = list(self.selected_features)
        if self.k_features is None or self.k_features >= len(all_feats):
            return self

        n_rows = len(df_train)
        if n_rows > self.max_mi_rows:
            rng = np.random.default_rng(self.seed)
            idx = np.sort(
                rng.choice(n_rows, size=self.max_mi_rows, replace=False)
            )
            df_sub = df_train.iloc[idx]
        else:
            df_sub = df_train

        x_tmp = df_sub[all_feats].copy()
        for col in all_feats:
            s_num = pd.to_numeric(x_tmp[col], errors='coerce')
            med = s_num.median()
            x_tmp[col] = s_num.fillna(0.0 if pd.isna(med) else med)

        y_arr = pd.to_numeric(
            df_sub['Target_Hours'], errors='coerce'
        ).to_numpy(dtype=np.float64)
        ok_y = np.isfinite(y_arr)
        x_mat = x_tmp.loc[ok_y].to_numpy(dtype=np.float64)
        y_vec = y_arr[ok_y]
        disc_mask = np.array(
            [c in DISCRETE_BASELINE_COLS for c in all_feats], dtype=bool
        )
        mi_vals = mutual_info_regression(
            x_mat,
            y_vec,
            discrete_features=disc_mask,
            random_state=self.seed,
        )
        self.mi_scores = {
            c: float(v) for c, v in zip(all_feats, mi_vals)
        }

        col_to_group: Dict[str, Tuple[str, ...]] = {}
        for grp in self.ATOMIC_GROUPS:
            present = tuple(c for c in grp if c in all_feats)
            for c in present:
                col_to_group[c] = present

        units: List[Tuple[float, Tuple[str, ...]]] = []
        seen_units = set()
        for col in all_feats:
            unit = col_to_group.get(col, (col,))
            if unit not in seen_units:
                seen_units.add(unit)
                score = max(self.mi_scores[c] for c in unit)
                units.append((score, unit))

        units.sort(key=lambda item: (-item[0], item[1][0]))
        chosen: List[str] = []
        for _, unit in units:
            if len(chosen) + len(unit) <= self.k_features or not chosen:
                chosen.extend(unit)
            if len(chosen) >= self.k_features:
                break

        self.selected_features = [c for c in all_feats if c in set(chosen)]
        assert_no_leakage_in_features(self.selected_features)
        return self

    def transform_y(self, y: pd.Series) -> np.ndarray:
        """Ap dung bien doi muc tieu log1p neu duoc cau hinh."""
        arr = np.asarray(y, dtype=np.float32)
        if self.target_transform == 'log1p':
            return np.log1p(np.maximum(arr, 0.0)).astype(np.float32)
        return arr

    def inverse_transform_y(self, y_pred: np.ndarray) -> np.ndarray:
        """Dao nguoc expm1 an toan ve don vi gio goc."""
        arr = np.asarray(y_pred, dtype=np.float64)
        if self.target_transform == 'log1p':
            clipped = np.clip(arr, -20.0, 10.0)
            return np.maximum(0.0, np.expm1(clipped)).astype(np.float32)
        return arr.astype(np.float32)


def make_m5_class(
    k_features: Optional[int] = None,
    target_transform: str = 'raw',
    name: str = 'M5',
) -> type:
    """Tao lop M5 khong-doi-so tuong thich giao dien run_expanding_window."""

    class _M5Variant(M5FeatureTarget):
        variant_name = name

        def __init__(self) -> None:
            super().__init__(
                k_features=k_features,
                target_transform=target_transform,
            )

    _M5Variant.__name__ = f'M5_{name}'
    return _M5Variant


# ======================================================================
# MODULE 6: HAM MAT MAT BAT DOI XUNG & DANH GIA PHAN TANG <=168H vs >168H
# ======================================================================


class Module6LossAndAntiBias:
    """M6 (Tran Hong Linh): Ham mat mat bat doi xung & danh gia phan tang."""

    def __init__(
        self,
        asym_alpha: float = 0.0,
        loss_type: str = 'quad',
        delta: float = 5.0,
        force_custom: bool = False,
    ) -> None:
        """Khoi tao ham loss (asym_alpha=0.0 & quad la Baseline doi xung)."""
        self.asym_alpha = float(asym_alpha)
        self.loss_type = str(loss_type)
        self.delta = float(delta)
        self.force_custom = bool(force_custom)

    @property
    def is_custom_objective(self) -> bool:
        """Xac dinh co su dung nhanh custom objective hay khong."""
        return (
            self.asym_alpha > 0.0
            or self.force_custom
            or self.loss_type != 'quad'
        )

    def numpy_loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """Tinh gia tri scalar loss khop 100% voi gradient/hessian."""
        err = np.asarray(y_pred, dtype=np.float64) - np.asarray(
            y_true, dtype=np.float64
        )
        w = np.where(err < 0.0, 1.0 + self.asym_alpha, 1.0)
        if self.loss_type == 'quad':
            per = err * err
        elif self.loss_type == 'huber':
            a = np.abs(err)
            d = self.delta
            per = np.where(a <= d, 0.5 * err * err, d * (a - 0.5 * d))
        elif self.loss_type == 'logcosh':
            a = np.abs(err)
            per = a + np.log1p(np.exp(-2.0 * a)) - np.log(2.0)
        else:
            raise ValueError(f'loss_type khong hop le: {self.loss_type}')
        return float(np.mean(w * per))

    def torch_loss(self, err: 'torch.Tensor') -> 'torch.Tensor':
        """Tinh loss PyTorch cho 4 mo hinh Deep Learning."""
        w = torch.where(
            err < 0.0,
            torch.full_like(err, 1.0 + self.asym_alpha),
            torch.ones_like(err),
        )
        if self.loss_type == 'quad':
            per = err ** 2
        elif self.loss_type == 'huber':
            a = err.abs()
            d = self.delta
            per = torch.where(a <= d, 0.5 * err * err, d * (a - 0.5 * d))
        elif self.loss_type == 'logcosh':
            a = err.abs()
            per = a + torch.nn.functional.softplus(-2.0 * a) - float(
                np.log(2.0)
            )
        else:
            raise ValueError(f'loss_type khong hop le: {self.loss_type}')
        return (w * per).mean()

    def make_lgb_xgb_objective(self) -> Optional[Callable]:
        """Tra ve ham (grad, hess) tuy bien hoac None neu la built-in."""
        if not self.is_custom_objective:
            return None
        alpha = self.asym_alpha
        kind = self.loss_type
        delta = self.delta

        def _obj(
            y_true: np.ndarray, y_pred: np.ndarray
        ) -> Tuple[np.ndarray, np.ndarray]:
            err = np.asarray(y_pred, dtype=np.float64) - np.asarray(
                y_true, dtype=np.float64
            )
            w = np.where(err < 0.0, 1.0 + alpha, 1.0)
            if kind == 'quad':
                return 2.0 * w * err, 2.0 * w
            if kind == 'huber':
                return w * np.clip(err, -delta, delta), w.copy()
            t = np.tanh(err)
            return w * t, w * np.maximum(1.0 - t * t, 1e-2)

        return _obj

    @staticmethod
    def _compute_stratum_metrics(
        yt: np.ndarray,
        yp: np.ndarray,
        total_n: int,
        lower: Optional[np.ndarray] = None,
        upper: Optional[np.ndarray] = None,
    ) -> Dict[str, float]:
        """Tinh 13 chi so tren mot tang du lieu (All / LE_168h / GT_168h)."""
        n_sub = len(yt)
        if n_sub == 0:
            return {k: np.nan for k in STRATUM_METRIC_KEYS} | {'N': 0}

        err = yp - yt
        abs_err = np.abs(err)
        ss_res = float(np.sum(err ** 2))
        ss_tot = float(np.sum((yt - np.mean(yt)) ** 2))
        r2_val = 1.0 - (ss_res / ss_tot) if ss_tot > 1e-12 else np.nan
        asym_25 = float(
            np.mean(np.where(err < 0.0, 2.5, 1.0) * (err ** 2))
        )

        picp = np.nan
        mpiw = np.nan
        if lower is not None and upper is not None:
            covered = (yt >= lower) & (yt <= upper)
            picp = float(np.mean(covered) * 100.0)
            mpiw = float(np.mean(upper - lower))

        return {
            'N': int(n_sub),
            'Share_pct': float(100.0 * n_sub / max(total_n, 1)),
            'MAE_hours': float(np.mean(abs_err)),
            'RMSE_hours': float(np.sqrt(np.mean(err ** 2))),
            'MedAE_hours': float(np.median(abs_err)),
            'R2': float(r2_val) if np.isfinite(r2_val) else np.nan,
            'MBE_bias_hours': float(np.mean(err)),
            'Underprediction_pct': float(np.mean(err < 0.0) * 100.0),
            'Overprediction_pct': float(np.mean(err > 0.0) * 100.0),
            'Within_3h_pct': float(np.mean(abs_err <= 3.0) * 100.0),
            'AsymCost_w2_5': asym_25,
            'PICP_90_pct': picp,
            'MPIW_hours': mpiw,
        }

    @classmethod
    def evaluate_horizon_stratified(
        cls,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        lower: Optional[np.ndarray] = None,
        upper: Optional[np.ndarray] = None,
    ) -> Dict[str, Dict[str, float]]:
        """Danh gia dong thoi 3 tang All, LE_168h va GT_168h."""
        yt = np.asarray(y_true, dtype=np.float64)
        yp = np.asarray(y_pred, dtype=np.float64)
        valid = np.isfinite(yt) & np.isfinite(yp)
        if lower is not None and upper is not None:
            low_a = np.asarray(lower, dtype=np.float64)
            up_a = np.asarray(upper, dtype=np.float64)
            valid = valid & np.isfinite(low_a) & np.isfinite(up_a)
            low_v = low_a[valid]
            up_v = up_a[valid]
        else:
            low_v = None
            up_v = None

        if not np.any(valid):
            raise ValueError('Khong co mau (y_true, y_pred) huu han!')

        yt_v = yt[valid]
        yp_v = yp[valid]
        tot_n = len(yt_v)
        le_mask = yt_v <= 168.0
        gt_mask = yt_v > 168.0

        return {
            'All': cls._compute_stratum_metrics(
                yt_v, yp_v, tot_n, low_v, up_v
            ),
            'LE_168h': cls._compute_stratum_metrics(
                yt_v[le_mask],
                yp_v[le_mask],
                tot_n,
                low_v[le_mask] if low_v is not None else None,
                up_v[le_mask] if up_v is not None else None,
            ),
            'GT_168h': cls._compute_stratum_metrics(
                yt_v[gt_mask],
                yp_v[gt_mask],
                tot_n,
                low_v[gt_mask] if low_v is not None else None,
                up_v[gt_mask] if up_v is not None else None,
            ),
        }


def make_m6_class(
    loss_type: str = 'quad',
    alpha: float = 0.0,
    delta: float = 5.0,
    force_custom: bool = False,
) -> type:
    """Tao lop M6 khong-doi-so de truyen vao run_expanding_window."""

    class _M6Variant(Module6LossAndAntiBias):
        variant_kind = loss_type
        variant_alpha = float(alpha)

        def __init__(self) -> None:
            super().__init__(
                asym_alpha=alpha,
                loss_type=loss_type,
                delta=delta,
                force_custom=force_custom,
            )

    _M6Variant.__name__ = f'M6_{loss_type}_a{alpha}'
    return _M6Variant


# ======================================================================
# MODULE 7: KHOANG SAI SO CONFORMAL (GLOBAL & ADAPTIVE CLUSTERING)
# ======================================================================


class Module7ClusteringIntervals:
    """M7 Baseline (Le Sy Liem): Khoang tin cay toan cuc 1 cap."""

    def __init__(self, alpha_interval: float = 0.10) -> None:
        self.alpha_interval = float(alpha_interval)
        self.q_low: float = -5.0
        self.q_high: float = 5.0

    def fit(
        self,
        df_val: pd.DataFrame,
        y_val_true: np.ndarray,
        y_val_pred: np.ndarray,
    ) -> 'Module7ClusteringIntervals':
        """Uoc luong phan vi sai so toan cuc tren InnerVal_raw_cal."""
        residuals = np.asarray(y_val_true, dtype=np.float64) - np.asarray(
            y_val_pred, dtype=np.float64
        )
        residuals = residuals[np.isfinite(residuals)]
        low_q = self.alpha_interval / 2.0
        high_q = 1.0 - (self.alpha_interval / 2.0)
        self.q_low = float(np.quantile(residuals, low_q))
        self.q_high = float(np.quantile(residuals, high_q))
        return self

    def predict_intervals(
        self,
        df_eval: pd.DataFrame,
        y_eval_pred: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Tinh can duoi/can tren (rang buoc >= 0) cho tap danh gia."""
        yp = np.asarray(y_eval_pred, dtype=np.float64)
        lower = np.maximum(0.0, yp + self.q_low)
        upper = np.maximum(lower, yp + self.q_high)
        return lower, upper

    @staticmethod
    def evaluate_intervals(
        y_true: np.ndarray,
        lower: np.ndarray,
        upper: np.ndarray,
    ) -> Dict[str, float]:
        """Tinh PICP_90_pct va MPIW_hours."""
        yt = np.asarray(y_true, dtype=np.float64)
        valid = np.isfinite(yt) & np.isfinite(lower) & np.isfinite(upper)
        yt_v = yt[valid]
        low_v = lower[valid]
        up_v = upper[valid]
        covered = (yt_v >= low_v) & (yt_v <= up_v)
        return {
            'PICP_90_pct': float(np.mean(covered) * 100.0),
            'MPIW_hours': float(np.mean(up_v - low_v)),
        }


class Module7AdaptiveClusteringIntervals(Module7ClusteringIntervals):
    """M7 Cai tien: 8 cum (4 chang cu ly x Is_Tanker) + Fallback."""

    STAGE_NAMES: List[str] = [
        'Stage_0_LE_25NM',
        'Stage_1_25_75NM',
        'Stage_2_75_250NM',
        'Stage_3_GT_250NM',
    ]

    def __init__(
        self,
        alpha_interval: float = 0.10,
        min_cluster_samples: int = 50,
    ) -> None:
        super().__init__(alpha_interval=alpha_interval)
        self.min_cluster_samples = int(min_cluster_samples)
        self.lookup_low = np.full(8, -5.0, dtype=np.float64)
        self.lookup_high = np.full(8, 5.0, dtype=np.float64)
        self.cluster_counts = np.zeros(8, dtype=np.int64)
        self.stage_low = np.full(4, -5.0, dtype=np.float64)
        self.stage_high = np.full(4, 5.0, dtype=np.float64)

    @staticmethod
    def _extract_cluster_ids(
        df: pd.DataFrame,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        n_rows = len(df)
        if 'Dist_To_Berth_NM' in df.columns:
            dist = np.nan_to_num(
                df['Dist_To_Berth_NM'].to_numpy(dtype=np.float64, copy=False),
                nan=150.0,
            )
        else:
            dist = np.full(n_rows, 150.0, dtype=np.float64)

        if 'Is_Tanker' in df.columns:
            tanker = np.nan_to_num(
                df['Is_Tanker'].to_numpy(dtype=np.float64, copy=False),
                nan=0.0,
            )
            t_flag = (tanker > 0.5).astype(np.int64)
        else:
            t_flag = np.zeros(n_rows, dtype=np.int64)

        stage_ids = np.clip(
            np.digitize(dist, bins=[25.0, 75.0, 250.0]), 0, 3
        ).astype(np.int64)
        cluster_ids = stage_ids * 2 + t_flag
        return stage_ids, t_flag, cluster_ids

    def fit(
        self,
        df_val: pd.DataFrame,
        y_val_true: np.ndarray,
        y_val_pred: np.ndarray,
    ) -> 'Module7AdaptiveClusteringIntervals':
        """Hoc phan vi 8 cum tren InnerVal_raw_cal voi Fallback da tang."""
        residuals = np.asarray(y_val_true, dtype=np.float64) - np.asarray(
            y_val_pred, dtype=np.float64
        )
        valid = np.isfinite(residuals)
        if not np.any(valid):
            raise ValueError('Khong co residual huu han trong InnerVal!')

        low_q = self.alpha_interval / 2.0
        high_q = 1.0 - (self.alpha_interval / 2.0)
        self.q_low = float(np.quantile(residuals[valid], low_q))
        self.q_high = float(np.quantile(residuals[valid], high_q))

        stage_ids, _, cluster_ids = self._extract_cluster_ids(df_val)
        for s_idx in range(4):
            s_mask = valid & (stage_ids == s_idx)
            if int(np.sum(s_mask)) >= self.min_cluster_samples:
                self.stage_low[s_idx] = float(
                    np.quantile(residuals[s_mask], low_q)
                )
                self.stage_high[s_idx] = float(
                    np.quantile(residuals[s_mask], high_q)
                )
            else:
                self.stage_low[s_idx] = self.q_low
                self.stage_high[s_idx] = self.q_high

        for c_idx in range(8):
            s_idx = c_idx // 2
            c_mask = valid & (cluster_ids == c_idx)
            cnt = int(np.sum(c_mask))
            self.cluster_counts[c_idx] = cnt
            if cnt >= self.min_cluster_samples:
                self.lookup_low[c_idx] = float(
                    np.quantile(residuals[c_mask], low_q)
                )
                self.lookup_high[c_idx] = float(
                    np.quantile(residuals[c_mask], high_q)
                )
            else:
                self.lookup_low[c_idx] = self.stage_low[s_idx]
                self.lookup_high[c_idx] = self.stage_high[s_idx]
        return self

    def predict_intervals(
        self,
        df_eval: pd.DataFrame,
        y_eval_pred: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Du bao can duoi va can tren theo 8 cum hanh vi."""
        yp = np.asarray(y_eval_pred, dtype=np.float64)
        _, _, cluster_ids = self._extract_cluster_ids(df_eval)
        lower = np.maximum(0.0, yp + self.lookup_low[cluster_ids])
        upper = np.maximum(lower, yp + self.lookup_high[cluster_ids])
        return lower, upper

    def evaluate_intervals_by_stage(
        self,
        df_eval: pd.DataFrame,
        y_true: np.ndarray,
        lower: np.ndarray,
        upper: np.ndarray,
    ) -> pd.DataFrame:
        """Danh gia PICP/MPIW theo 4 chang & 8 cum tren 3 tang."""
        stage_ids, _, cluster_ids = self._extract_cluster_ids(df_eval)
        yt = np.asarray(y_true, dtype=np.float64)
        low_a = np.asarray(lower, dtype=np.float64)
        up_a = np.asarray(upper, dtype=np.float64)
        valid = np.isfinite(yt) & np.isfinite(low_a) & np.isfinite(up_a)

        records: List[Dict[str, Any]] = []
        strata = [
            ('All', valid),
            ('LE_168h', valid & (yt <= 168.0)),
            ('GT_168h', valid & (yt > 168.0)),
        ]
        for s_idx in range(4):
            s_name = self.STAGE_NAMES[s_idx]
            for h_name, h_mask in strata:
                m_st = h_mask & (stage_ids == s_idx)
                cnt = int(np.sum(m_st))
                if cnt > 0:
                    cov = (yt[m_st] >= low_a[m_st]) & (yt[m_st] <= up_a[m_st])
                    records.append({
                        'Level': 'Stage',
                        'Stage': s_name,
                        'Is_Tanker': -1,
                        'Horizon': h_name,
                        'N': cnt,
                        'PICP_90_pct': float(np.mean(cov) * 100.0),
                        'MPIW_hours': float(np.mean(up_a[m_st] - low_a[m_st])),
                    })
            for t_flag in (0, 1):
                c_idx = s_idx * 2 + t_flag
                for h_name, h_mask in strata:
                    m_cl = h_mask & (cluster_ids == c_idx)
                    cnt_c = int(np.sum(m_cl))
                    if cnt_c > 0:
                        cov_c = (yt[m_cl] >= low_a[m_cl]) & (
                            yt[m_cl] <= up_a[m_cl]
                        )
                        records.append({
                            'Level': 'Cluster',
                            'Stage': s_name,
                            'Is_Tanker': t_flag,
                            'Horizon': h_name,
                            'N': cnt_c,
                            'PICP_90_pct': float(np.mean(cov_c) * 100.0),
                            'MPIW_hours': float(
                                np.mean(up_a[m_cl] - low_a[m_cl])
                            ),
                        })
        return pd.DataFrame(records)


# ======================================================================
# 3 GBDT + 4 PYTORCH DL (ZERO-OOM, FIXED ST-GNN & INIT/BASE SCORE)
# ======================================================================


def _build_dl_modules_if_torch_available() -> Dict[str, type]:
    if torch is None:
        return {}

    class BiLSTMRegressor(nn.Module):
        """Mang hoi quy Bi-LSTM 2 chieu cho quy dao AIS."""

        def __init__(self, in_dim: int, hidden_dim: int = 64) -> None:
            super().__init__()
            self.lstm = nn.LSTM(
                in_dim,
                hidden_dim,
                num_layers=2,
                batch_first=True,
                bidirectional=True,
                dropout=0.1,
            )
            self.head = nn.Sequential(
                nn.Linear(hidden_dim * 2, 32),
                nn.ReLU(),
                nn.Linear(32, 1),
            )

        def forward(self, x: 'torch.Tensor') -> 'torch.Tensor':
            out, _ = self.lstm(x)
            return self.head(out[:, -1, :]).squeeze(-1)

    class PatchTSTRegressor(nn.Module):
        """Transformer phan doan chuoi thoi gian (PatchTST)."""

        def __init__(
            self,
            in_dim: int,
            seq_len: int = 12,
            patch_len: int = 4,
            d_model: int = 64,
            nhead: int = 4,
        ) -> None:
            super().__init__()
            self.patch_len = patch_len
            self.num_patches = seq_len // patch_len
            self.proj = nn.Linear(patch_len * in_dim, d_model)
            self.pos_emb = nn.Parameter(
                torch.zeros(1, self.num_patches, d_model)
            )
            enc_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=128,
                dropout=0.1,
                batch_first=True,
            )
            self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
            self.head = nn.Linear(d_model, 1)

        def forward(self, x: 'torch.Tensor') -> 'torch.Tensor':
            bsz = x.shape[0]
            x_p = x.reshape(bsz, self.num_patches, -1)
            z = self.encoder(self.proj(x_p) + self.pos_emb)
            return self.head(z.mean(dim=1)).squeeze(-1)

    class TFTRegressor(nn.Module):
        """Temporal Fusion Transformer rut gon cho AIS."""

        def __init__(
            self, in_dim: int, hidden_dim: int = 64, nhead: int = 4
        ) -> None:
            super().__init__()
            self.vsn = nn.Sequential(
                nn.Linear(in_dim, in_dim), nn.Softmax(dim=-1)
            )
            self.lstm = nn.LSTM(in_dim, hidden_dim, batch_first=True)
            self.attn = nn.MultiheadAttention(
                hidden_dim, nhead, dropout=0.1, batch_first=True
            )
            self.norm = nn.LayerNorm(hidden_dim)
            self.head = nn.Linear(hidden_dim, 1)

        def forward(self, x: 'torch.Tensor') -> 'torch.Tensor':
            x_sel = x * self.vsn(x)
            h, _ = self.lstm(x_sel)
            attn_out, _ = self.attn(h, h, h)
            z = self.norm(h[:, -1, :] + attn_out[:, -1, :])
            return self.head(z).squeeze(-1)

    class STGNNRegressor(nn.Module):
        """ST-GNN da va LayerNorm + Residual quanh khoi do thi batch."""

        def __init__(self, in_dim: int, hidden_dim: int = 64) -> None:
            super().__init__()
            self.gru = nn.GRU(in_dim, hidden_dim, batch_first=True)
            self.gnn_proj = nn.Linear(hidden_dim, hidden_dim)
            self.gnn_drop = nn.Dropout(0.1)
            self.norm_node = nn.LayerNorm(hidden_dim)
            self.head = nn.Sequential(
                nn.Linear(hidden_dim * 2, 32),
                nn.ReLU(),
                nn.Linear(32, 1),
            )

        def forward(self, x: 'torch.Tensor') -> 'torch.Tensor':
            h_seq, _ = self.gru(x)
            h_node = h_seq[:, -1, :]
            h_norm = torch.nn.functional.normalize(h_node, p=2, dim=-1)
            adj = torch.softmax(torch.matmul(h_norm, h_norm.T), dim=-1)
            agg_msg = torch.matmul(adj, self.gnn_proj(h_node))
            h_out = self.norm_node(h_node + self.gnn_drop(agg_msg))
            combined = torch.cat([h_node, h_out], dim=-1)
            return self.head(combined).squeeze(-1)

    return {
        'Bi-LSTM': BiLSTMRegressor,
        'PatchTST': PatchTSTRegressor,
        'TFT': TFTRegressor,
        'ST-GNN': STGNNRegressor,
    }


DL_MODEL_REGISTRY = _build_dl_modules_if_torch_available()


def build_causal_window_indices(
    df_meta: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Tao anh xa chi so nhan qua 1D ton O(N) int32."""
    n_rows = len(df_meta)
    trip_codes = pd.factorize(df_meta['trip_id'], sort=False)[0]
    time_ns = (
        pd.to_datetime(df_meta['base_date_time'])
        .to_numpy(dtype='datetime64[ns]')
        .view(np.int64)
    )
    order = np.lexsort((time_ns, trip_codes)).astype(np.int32)
    inv_order = np.empty(n_rows, dtype=np.int32)
    inv_order[order] = np.arange(n_rows, dtype=np.int32)

    sorted_t = trip_codes[order]
    splits = np.flatnonzero(np.diff(sorted_t)) + 1
    bounds = np.concatenate(([0], splits, [n_rows]))
    trip_starts = np.zeros(n_rows, dtype=np.int32)
    for k in range(len(bounds) - 1):
        trip_starts[bounds[k]:bounds[k + 1]] = bounds[k]
    return order, inv_order, trip_starts


class LazyCausalTripDataset(Dataset):
    """Dataset cat cua so nhan qua 12 buoc truc tiep tren mang 2D."""

    def __init__(
        self,
        x_2d_sorted: np.ndarray,
        trip_starts: np.ndarray,
        sample_positions: np.ndarray,
        y_sorted: Optional[np.ndarray] = None,
        seq_len: int = 12,
    ) -> None:
        self.x_2d = x_2d_sorted
        self.trip_starts = trip_starts
        self.positions = np.asarray(sample_positions, dtype=np.int32)
        self.y_sorted = y_sorted
        self.seq_len = seq_len
        self.offsets = np.arange(-(seq_len - 1), 1, dtype=np.int32)[None, :]

    def __len__(self) -> int:
        return len(self.positions)

    def __getitem__(self, idx: int) -> Any:
        pos = int(self.positions[idx])
        idxs = np.maximum(pos + self.offsets[0], self.trip_starts[pos])
        wx = self.x_2d[idxs]
        if self.y_sorted is None:
            return wx
        return wx, self.y_sorted[pos]

    def fetch_batch(self, batch_pos: np.ndarray) -> Any:
        """Trich xuat vector hoa nguyen mini-batch o toc do C."""
        idx_2d = np.maximum(
            batch_pos[:, None] + self.offsets,
            self.trip_starts[batch_pos, None],
        )
        bx = self.x_2d[idx_2d]
        if self.y_sorted is None:
            return bx
        return bx, self.y_sorted[batch_pos]


def _predict_lazy_dl(
    model: 'nn.Module',
    ds_obj: LazyCausalTripDataset,
    eval_bs: int,
    device: 'torch.device',
    is_stgnn: bool,
    seed: int = SEED,
) -> np.ndarray:
    """Du bao Zero-OOM; hoan vi xac dinh theo seed neu la ST-GNN."""
    n_pos = len(ds_obj.positions)
    if n_pos == 0:
        return np.array([], dtype=np.float32)

    if is_stgnn:
        perm = np.random.default_rng(seed).permutation(n_pos)
        pos_iter = ds_obj.positions[perm]
    else:
        perm = None
        pos_iter = ds_obj.positions

    preds_chunks: List[np.ndarray] = []
    model.eval()
    with torch.no_grad():
        for s_i in range(0, n_pos, eval_bs):
            b_pos = pos_iter[s_i:s_i + eval_bs]
            out_b = ds_obj.fetch_batch(b_pos)
            bx_np = out_b[0] if isinstance(out_b, tuple) else out_b
            bx = torch.from_numpy(bx_np).to(device)
            preds_chunks.append(model(bx).cpu().numpy())

    joined = np.concatenate(preds_chunks).astype(np.float32)
    if perm is not None:
        out_arr = np.empty(n_pos, dtype=np.float32)
        out_arr[perm] = joined
        return out_arr
    return joined


def _train_dl_regressor(
    model_name: str,
    df_fit_meta: pd.DataFrame,
    x_fit: pd.DataFrame,
    y_fit: np.ndarray,
    df_val_clean_meta: pd.DataFrame,
    x_val_clean: pd.DataFrame,
    y_val_clean: np.ndarray,
    df_val_cal_meta: pd.DataFrame,
    x_val_cal: pd.DataFrame,
    df_eval_meta: pd.DataFrame,
    x_eval: pd.DataFrame,
    m6_loss_mod: Module6LossAndAntiBias,
    seq_len: int = 12,
    epochs: int = 15,
    batch_size: int = 512,
    max_windows: Optional[int] = MAX_DL_WINDOWS_PER_SPLIT,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Huan luyen PyTorch DL va tra ve (vp_clean, vp_cal, ep_eval)."""
    if torch is None or model_name not in DL_MODEL_REGISTRY:
        raise RuntimeError(f'Khong co PyTorch hoac mo hinh {model_name}!')

    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    device = torch.device('cuda' if HAS_CUDA else 'cpu')

    fit_meds = x_fit.median().fillna(0.0)
    xf_a = x_fit.fillna(fit_meds).to_numpy(dtype=np.float32)
    xvc_a = x_val_clean.fillna(fit_meds).to_numpy(dtype=np.float32)
    xvcal_a = x_val_cal.fillna(fit_meds).to_numpy(dtype=np.float32)
    xe_a = x_eval.fillna(fit_meds).to_numpy(dtype=np.float32)

    mean_v = xf_a.mean(axis=0, keepdims=True)
    std_raw = xf_a.std(axis=0, keepdims=True)
    std_v = np.where(std_raw < 1e-6, 1.0, std_raw)

    def _scale(arr: np.ndarray) -> np.ndarray:
        return np.nan_to_num(
            ((arr - mean_v) / std_v).astype(np.float32),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

    xf_s, xvc_s, xvcal_s, xe_s = (
        _scale(xf_a),
        _scale(xvc_a),
        _scale(xvcal_a),
        _scale(xe_a),
    )

    ord_f, _, ts_f = build_causal_window_indices(df_fit_meta)
    ord_vc, inv_vc, ts_vc = build_causal_window_indices(df_val_clean_meta)
    ord_vcal, inv_vcal, ts_vcal = build_causal_window_indices(df_val_cal_meta)
    ord_e, inv_e, ts_e = build_causal_window_indices(df_eval_meta)

    fit_pos = np.arange(len(df_fit_meta), dtype=np.int32)
    if max_windows is not None and len(fit_pos) > max_windows:
        fit_pos = rng.choice(
            fit_pos, size=max_windows, replace=False
        ).astype(np.int32)

    ds_fit = LazyCausalTripDataset(
        xf_s[ord_f],
        ts_f,
        fit_pos,
        y_sorted=np.asarray(y_fit, dtype=np.float32)[ord_f],
        seq_len=seq_len,
    )
    ds_val_c = LazyCausalTripDataset(
        xvc_s[ord_vc],
        ts_vc,
        np.arange(len(df_val_clean_meta), dtype=np.int32),
        y_sorted=np.asarray(y_val_clean, dtype=np.float32)[ord_vc],
        seq_len=seq_len,
    )
    ds_val_cal = LazyCausalTripDataset(
        xvcal_s[ord_vcal],
        ts_vcal,
        np.arange(len(df_val_cal_meta), dtype=np.int32),
        y_sorted=None,
        seq_len=seq_len,
    )
    ds_eval = LazyCausalTripDataset(
        xe_s[ord_e],
        ts_e,
        np.arange(len(df_eval_meta), dtype=np.int32),
        y_sorted=None,
        seq_len=seq_len,
    )

    in_dim = x_fit.shape[1]
    cls_m = DL_MODEL_REGISTRY[model_name]
    if model_name == 'PatchTST':
        model = cls_m(in_dim=in_dim, seq_len=seq_len).to(device)
    else:
        model = cls_m(in_dim=in_dim).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-3, weight_decay=1e-4
    )
    is_stgnn = model_name == 'ST-GNN'
    eval_bs = batch_size if is_stgnn else batch_size * 2
    y_vc_sorted = np.asarray(y_val_clean, dtype=np.float32)[ord_vc]
    use_custom = m6_loss_mod.is_custom_objective

    best_score = float('inf')
    best_state = None
    patience = 3
    no_improve = 0

    for _ in range(epochs):
        model.train()
        shuffled = rng.permutation(fit_pos)
        for s_i in range(0, len(shuffled), batch_size):
            b_pos = shuffled[s_i:s_i + batch_size]
            bx_np, by_np = ds_fit.fetch_batch(b_pos)
            bx = torch.from_numpy(bx_np).to(device)
            by = torch.from_numpy(by_np).to(device)
            optimizer.zero_grad()
            loss = m6_loss_mod.torch_loss(model(bx) - by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        vp_sorted = _predict_lazy_dl(
            model, ds_val_c, eval_bs, device, is_stgnn=is_stgnn, seed=SEED
        )
        if use_custom:
            val_score = m6_loss_mod.numpy_loss(y_vc_sorted, vp_sorted)
        else:
            val_score = float(np.mean(np.abs(vp_sorted - y_vc_sorted)))

        if val_score < best_score:
            best_score = val_score
            best_state = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

    vp_clean = _predict_lazy_dl(
        model, ds_val_c, eval_bs, device, is_stgnn=is_stgnn, seed=SEED
    )[inv_vc]
    vp_cal = _predict_lazy_dl(
        model, ds_val_cal, eval_bs, device, is_stgnn=is_stgnn, seed=SEED + 1
    )[inv_vcal]
    ep_eval = _predict_lazy_dl(
        model, ds_eval, eval_bs, device, is_stgnn=is_stgnn, seed=SEED + 2
    )[inv_e]

    del model, optimizer, best_state, ds_fit, ds_val_c, ds_val_cal, ds_eval
    if HAS_CUDA:
        torch.cuda.empty_cache()
    gc.collect()
    return vp_clean, vp_cal, ep_eval


def train_and_predict_model(
    model_name: str,
    df_fit_meta: pd.DataFrame,
    x_fit: pd.DataFrame,
    y_fit: np.ndarray,
    df_val_clean_meta: pd.DataFrame,
    x_val_clean: pd.DataFrame,
    y_val_clean: np.ndarray,
    df_val_cal_meta: pd.DataFrame,
    x_val_cal: pd.DataFrame,
    df_eval_meta: pd.DataFrame,
    x_eval: pd.DataFrame,
    m6_loss_mod: Module6LossAndAntiBias,
    dl_epochs: int = 15,
    dl_batch_size: int = 512,
    max_dl_windows: Optional[int] = MAX_DL_WINDOWS_PER_SPLIT,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Huan luyen GBDT / PyTorch va tra ve (vp_clean, vp_cal, ep_eval)."""
    custom_obj = m6_loss_mod.make_lgb_xgb_objective()
    is_custom = m6_loss_mod.is_custom_objective

    if model_name == 'LightGBM':
        if lgb is None:
            raise RuntimeError('Thieu thu vien lightgbm!')
        params: Dict[str, Any] = {
            'n_estimators': 1000,
            'learning_rate': 0.05,
            'num_leaves': 31,
            'subsample_for_bin': 50000,
            'random_state': SEED,
            'n_jobs': -1,
            'verbosity': -1,
        }
        if custom_obj is not None:
            base = float(np.mean(y_fit))
            params['objective'] = custom_obj

            def _lgb_feval(yt: np.ndarray, yp: np.ndarray):
                return (
                    'asym_loss',
                    m6_loss_mod.numpy_loss(yt, yp),
                    False,
                )

            model = lgb.LGBMRegressor(**params)
            model.fit(
                x_fit,
                y_fit,
                init_score=np.full(len(x_fit), base, dtype=np.float64),
                eval_set=[(x_val_clean, y_val_clean)],
                eval_init_score=[
                    np.full(len(x_val_clean), base, dtype=np.float64)
                ],
                eval_metric=_lgb_feval,
                callbacks=[lgb.early_stopping(50, verbose=False)],
            )
            return (
                model.predict(x_val_clean) + base,
                model.predict(x_val_cal) + base,
                model.predict(x_eval) + base,
            )

        model = lgb.LGBMRegressor(**params)
        model.fit(
            x_fit,
            y_fit,
            eval_set=[(x_val_clean, y_val_clean)],
            eval_metric='l1',
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        return (
            model.predict(x_val_clean),
            model.predict(x_val_cal),
            model.predict(x_eval),
        )

    if model_name == 'XGBoost':
        if xgb is None:
            raise RuntimeError('Thieu thu vien xgboost!')
        params_x: Dict[str, Any] = {
            'n_estimators': 1000,
            'learning_rate': 0.05,
            'max_depth': 6,
            'tree_method': 'hist',
            'random_state': SEED,
            'n_jobs': -1,
            'early_stopping_rounds': 50,
        }
        if custom_obj is not None:
            base = float(np.mean(y_fit))
            params_x['objective'] = custom_obj
            params_x['base_score'] = base

            def _xgb_metric(yt: np.ndarray, yp: np.ndarray) -> float:
                return m6_loss_mod.numpy_loss(yt, yp)

            params_x['eval_metric'] = _xgb_metric
        else:
            params_x['eval_metric'] = 'mae'

        devs = ['cuda', 'cpu'] if HAS_CUDA else ['cpu']
        model = None
        for dev in devs:
            try:
                p_try = dict(params_x, device=dev)
                model = xgb.XGBRegressor(**p_try)
                model.fit(
                    x_fit,
                    y_fit,
                    eval_set=[(x_val_clean, y_val_clean)],
                    verbose=False,
                )
                break
            except Exception:
                model = None
        if model is None:
            p_fb = dict(params_x, device='cpu', eval_metric='mae')
            model = xgb.XGBRegressor(**p_fb)
            model.fit(
                x_fit,
                y_fit,
                eval_set=[(x_val_clean, y_val_clean)],
                verbose=False,
            )
        return (
            model.predict(x_val_clean),
            model.predict(x_val_cal),
            model.predict(x_eval),
        )

    if model_name == 'CatBoost':
        if CatBoostRegressor is None:
            raise RuntimeError('Thieu thu vien catboost!')
        if is_custom:
            if m6_loss_mod.loss_type != 'quad':
                raise NotImplementedError(
                    'CatBoost chi ho tro Expectile cho loss_type="quad".'
                )
            tau = (1.0 + m6_loss_mod.asym_alpha) / (
                2.0 + m6_loss_mod.asym_alpha
            )
            loss_fn = f'Expectile:alpha={tau:.6f}'
        else:
            loss_fn = 'RMSE'

        params_c: Dict[str, Any] = {
            'iterations': 1000,
            'learning_rate': 0.05,
            'depth': 6,
            'loss_function': loss_fn,
            'eval_metric': 'MAE',
            'random_seed': SEED,
            'verbose': 0,
        }
        gpu_flags = [True, False] if (HAS_CUDA and not is_custom) else [False]
        model = None
        for use_gpu in gpu_flags:
            p_try = dict(params_c)
            if use_gpu:
                p_try['task_type'] = 'GPU'
            else:
                p_try['task_type'] = 'CPU'
                p_try['thread_count'] = -1
            try:
                model = CatBoostRegressor(**p_try)
                model.fit(
                    x_fit,
                    y_fit,
                    eval_set=(x_val_clean, y_val_clean),
                    early_stopping_rounds=50,
                    verbose=False,
                )
                break
            except Exception:
                if not use_gpu:
                    raise
                model = None
        return (
            model.predict(x_val_clean),
            model.predict(x_val_cal),
            model.predict(x_eval),
        )

    if model_name in ('Bi-LSTM', 'PatchTST', 'TFT', 'ST-GNN'):
        return _train_dl_regressor(
            model_name=model_name,
            df_fit_meta=df_fit_meta,
            x_fit=x_fit,
            y_fit=y_fit,
            df_val_clean_meta=df_val_clean_meta,
            x_val_clean=x_val_clean,
            y_val_clean=y_val_clean,
            df_val_cal_meta=df_val_cal_meta,
            x_val_cal=x_val_cal,
            df_eval_meta=df_eval_meta,
            x_eval=x_eval,
            m6_loss_mod=m6_loss_mod,
            epochs=dl_epochs,
            batch_size=dl_batch_size,
            max_windows=max_dl_windows,
        )

    raise ValueError(f'Mo hinh khong nam trong danh sach chot: {model_name}')


# ======================================================================
# DONG CO THUC NGHIEM EXPANDING WINDOW V2 + CHECKPOINTING + ZIP
# ======================================================================


def _flatten_horizon_metrics(
    strat: Dict[str, Dict[str, float]],
) -> Dict[str, float]:
    """Lam phang chi so All + LE168_* + GT168_* vao 1 dong folds.csv."""
    flat: Dict[str, float] = {'Stratum': 'All'}
    for key in STRATUM_METRIC_KEYS:
        flat[key] = strat['All'][key]
    for prefix, h_key in (('LE168', 'LE_168h'), ('GT168', 'GT_168h')):
        for key in STRATUM_METRIC_KEYS:
            flat[f'{prefix}_{key}'] = strat[h_key][key]
    return flat


def run_expanding_window_experiment(
    experiment_name: str = 'BASELINE_7_MODULES',
    variant_name: str = 'default',
    active_models: Optional[List[str]] = None,
    m1_cls: type = Module1OutlierFilter,
    m2_cls: type = Module2Imputer,
    m3_cls: type = Module3FeatureCreator,
    m4_cls: type = Module4WeatherEnricher,
    m5_cls: type = Module5FeatureTargetSelector,
    m6_cls: type = Module6LossAndAntiBias,
    m7_cls: type = Module7ClusteringIntervals,
    version_name: str = '03_ban_tien_xu_ly_27_thuoc_tinh',
    quarter_cache: Optional[Dict[str, pd.DataFrame]] = None,
    fold_specs: Optional[List[Dict[str, Any]]] = None,
    work_dir: str = './results_v2',
    resume_from_disk: bool = RESUME_FROM_DISK,
    dl_epochs: int = 15,
    dl_batch_size: int = 512,
    max_dl_windows: Optional[int] = MAX_DL_WINDOWS_PER_SPLIT,
    save_eval_preds: bool = False,
) -> Dict[str, pd.DataFrame]:
    """Chay thuc nghiem Expanding Window v2 va xuat day du 4 bang CSV."""
    os.makedirs(work_dir, exist_ok=True)
    slug = f'{experiment_name}_{variant_name}'.lower()
    done_flag = os.path.join(work_dir, f'{slug}_done.flag')
    p_folds = os.path.join(work_dir, f'{slug}_folds.csv')
    p_summary = os.path.join(work_dir, f'{slug}_summary.csv')
    p_horizons = os.path.join(work_dir, f'{slug}_horizons.csv')
    p_clusters = os.path.join(work_dir, f'{slug}_m7_clusters.csv')

    if (
        resume_from_disk
        and os.path.isfile(done_flag)
        and os.path.isfile(p_folds)
        and os.path.isfile(p_summary)
        and os.path.isfile(p_horizons)
    ):
        print(f'[RESUME SKIP] {experiment_name}/{variant_name} da xong.')
        return {
            'folds': pd.read_csv(p_folds),
            'summary': pd.read_csv(p_summary),
            'horizons': pd.read_csv(p_horizons),
            'clusters': (
                pd.read_csv(p_clusters)
                if os.path.isfile(p_clusters)
                else pd.DataFrame()
            ),
        }

    models_to_run = (
        list(active_models)
        if active_models is not None
        else list(ACTIVE_MODELS)
    )
    folds = fold_specs if fold_specs is not None else EXPANDING_FOLDS
    q_needed = sorted({
        q for sp in folds for q in [*sp['train_quarters'], sp['eval_quarter']]
    })
    q_cache = (
        quarter_cache
        if quarter_cache is not None
        else load_quarter_cache(quarters=q_needed, version_name=version_name)
    )
    if hasattr(m4_cls, 'warm_cache_from_quarters'):
        m4_cls.warm_cache_from_quarters(q_cache)

    fold_rows: List[Dict[str, Any]] = []
    horizon_rows: List[Dict[str, Any]] = []
    cluster_rows: List[pd.DataFrame] = []

    for f_spec in folds:
        f_name = f_spec['fold']
        tr_qs = f_spec['train_quarters']
        ev_q = f_spec['eval_quarter']
        print(f'\n--- {experiment_name}/{variant_name} | {f_name}: '
              f'{"+".join(tr_qs)} -> {ev_q} ---')

        df_tr_raw = pd.concat([q_cache[q] for q in tr_qs], ignore_index=True)
        df_ev_raw = q_cache[ev_q].copy()
        df_ev_raw['_eval_row_id'] = np.arange(len(df_ev_raw), dtype=np.int64)
        df_ev_ref = Module1OutlierFilter().transform_eval(df_ev_raw)
        ref_rows = len(df_ev_ref)
        ref_trips = (
            int(df_ev_ref['trip_id'].nunique()) if ref_rows > 0 else 0
        )
        del df_ev_ref

        df_fit_raw, df_val_raw = split_train_inner_val_by_trip(df_tr_raw)
        del df_tr_raw

        # M1: Fit & Val-clean loc implied_spd; Val-cal & Eval giu nguyen tho.
        m1 = m1_cls()
        df_fit_1 = m1.fit_transform_train(df_fit_raw)
        df_val_clean_1 = m1.fit_transform_train(df_val_raw)
        df_val_cal_1 = m1.transform_eval(df_val_raw)
        df_ev_1 = m1.transform_eval(df_ev_raw)
        del df_fit_raw, df_val_raw, df_ev_raw

        ev_trips = int(df_ev_1['trip_id'].nunique()) if len(df_ev_1) > 0 else 0
        cov_rows_pct = 100.0 * len(df_ev_1) / max(ref_rows, 1)
        cov_trips_pct = 100.0 * ev_trips / max(ref_trips, 1)

        # M2: Fit duy nhat tren df_fit_1 -> transform sang 3 tap con lai.
        m2 = m2_cls().fit(df_fit_1)
        df_fit_2 = m2.transform(df_fit_1)
        df_val_clean_2 = m2.transform(df_val_clean_1)
        df_val_cal_2 = m2.transform(df_val_cal_1)
        df_ev_2 = m2.transform(df_ev_1)
        del df_fit_1, df_val_clean_1, df_val_cal_1, df_ev_1

        # M3: Tao dac trung nhan qua.
        m3 = m3_cls().fit(df_fit_2)
        df_fit_3 = m3.transform(df_fit_2)
        df_val_clean_3 = m3.transform(df_val_clean_2)
        df_val_cal_3 = m3.transform(df_val_cal_2)
        df_ev_3 = m3.transform(df_ev_2)
        del df_fit_2, df_val_clean_2, df_val_cal_2, df_ev_2

        # M4: Lam giau thoi tiet RTMA.
        m4 = m4_cls().fit(df_fit_3)
        df_fit_4 = m4.transform(df_fit_3)
        df_val_clean_4 = m4.transform(df_val_clean_3)
        df_val_cal_4 = m4.transform(df_val_cal_3)
        df_ev_4 = m4.transform(df_ev_3)
        del df_fit_3, df_val_clean_3, df_val_cal_3, df_ev_3

        # M5: Chon dac trung & bien doi Target.
        m5 = m5_cls().fit(df_fit_4)
        x_fit = m5.transform_x(df_fit_4)
        x_val_clean = m5.transform_x(df_val_clean_4)
        x_val_cal = m5.transform_x(df_val_cal_4)
        x_eval = m5.transform_x(df_ev_4)

        y_fit_tr = m5.transform_y(df_fit_4['Target_Hours'])
        y_vc_tr = m5.transform_y(df_val_clean_4['Target_Hours'])
        y_vcal_true = df_val_cal_4['Target_Hours'].to_numpy(dtype=np.float64)
        y_eval_true = df_ev_4['Target_Hours'].to_numpy(dtype=np.float64)

        m6 = m6_cls()

        for m_name in models_to_run:
            t_m = time.time()
            _, vp_cal_raw, ep_raw = train_and_predict_model(
                model_name=m_name,
                df_fit_meta=df_fit_4,
                x_fit=x_fit,
                y_fit=y_fit_tr,
                df_val_clean_meta=df_val_clean_4,
                x_val_clean=x_val_clean,
                y_val_clean=y_vc_tr,
                df_val_cal_meta=df_val_cal_4,
                x_val_cal=x_val_cal,
                df_eval_meta=df_ev_4,
                x_eval=x_eval,
                m6_loss_mod=m6,
                dl_epochs=dl_epochs,
                dl_batch_size=dl_batch_size,
                max_dl_windows=max_dl_windows,
            )
            vp_cal_hrs = m5.inverse_transform_y(vp_cal_raw)
            ep_hrs = m5.inverse_transform_y(ep_raw)

            # M7 fit tren df_val_cal_4 (raw tail) o thang gio goc.
            m7 = m7_cls().fit(df_val_cal_4, y_vcal_true, vp_cal_hrs)
            low_b, high_b = m7.predict_intervals(df_ev_4, ep_hrs)

            strat = m6.evaluate_horizon_stratified(
                y_eval_true, ep_hrs, lower=low_b, upper=high_b
            )
            flat_m = _flatten_horizon_metrics(strat)
            train_sec = round(time.time() - t_m, 2)

            base_info = {
                'Experiment': experiment_name,
                'Variant': variant_name,
                'Fold': f_name,
                'Train_Q': '+'.join(tr_qs),
                'Eval_Q': ev_q,
                'Model': m_name,
                'Num_Features': int(x_fit.shape[1]),
                'Fit_Rows': int(len(x_fit)),
                'ValClean_Rows': int(len(x_val_clean)),
                'ValCal_Rows': int(len(x_val_cal)),
                'Eval_Ref_Rows': int(ref_rows),
                'Eval_Ref_Trips': int(ref_trips),
                'Eval_Rows': int(len(x_eval)),
                'Eval_Trips': int(ev_trips),
                'Eval_Row_Coverage_pct': round(cov_rows_pct, 4),
                'Eval_Trip_Coverage_pct': round(cov_trips_pct, 4),
                'Train_Sec': train_sec,
            }
            if save_eval_preds:
                p_dir = os.path.join(work_dir, 'preds')
                os.makedirs(p_dir, exist_ok=True)
                p_pq = os.path.join(p_dir, f'{slug}_{f_name}_{m_name}.parquet')
                pd.DataFrame({
                    'row_id': df_ev_4['_eval_row_id'].to_numpy(dtype=np.int64),
                    'trip_id': df_ev_4['trip_id'].astype(str).to_numpy(),
                    'y_true': y_eval_true.astype(np.float32),
                    'y_pred': ep_hrs.astype(np.float32),
                }).to_parquet(p_pq, index=False)
            fold_rows.append({**base_info, **flat_m})

            for h_key in ('All', 'LE_168h', 'GT_168h'):
                horizon_rows.append({
                    **base_info,
                    'Horizon': h_key,
                    **strat[h_key],
                })

            if hasattr(m7, 'evaluate_intervals_by_stage'):
                df_cl = m7.evaluate_intervals_by_stage(
                    df_ev_4, y_eval_true, low_b, high_b
                )
                if not df_cl.empty:
                    df_cl.insert(0, 'Model', m_name)
                    df_cl.insert(0, 'Fold', f_name)
                    df_cl.insert(0, 'Variant', variant_name)
                    df_cl.insert(0, 'Experiment', experiment_name)
                    cluster_rows.append(df_cl)

            print(
                f'  * [{m_name:8s}] All_MAE={strat["All"]["MAE_hours"]:6.2f}h'
                f' | LE168_MAE={strat["LE_168h"]["MAE_hours"]:5.2f}h'
                f' | GT168_MAE={strat["GT_168h"]["MAE_hours"]:6.2f}h'
                f' | PICP={strat["All"]["PICP_90_pct"]:5.1f}%'
                f' ({train_sec:.1f}s)'
            )

        gc.collect()

    df_folds = pd.DataFrame(fold_rows)
    df_horizons = pd.DataFrame(horizon_rows)
    num_cols = [
        c
        for c in df_folds.columns
        if c
        not in (
            'Experiment',
            'Variant',
            'Fold',
            'Train_Q',
            'Eval_Q',
            'Model',
            'Stratum',
        )
    ]
    df_summary = (
        df_folds.groupby(['Experiment', 'Variant', 'Model'], as_index=False)[
            num_cols
        ]
        .mean()
        .round(4)
    )
    df_summary.insert(3, 'Stratum', 'All')

    df_clusters = (
        pd.concat(cluster_rows, ignore_index=True)
        if cluster_rows
        else pd.DataFrame()
    )

    df_folds.to_csv(p_folds, index=False)
    df_summary.to_csv(p_summary, index=False)
    df_horizons.to_csv(p_horizons, index=False)
    if not df_clusters.empty:
        df_clusters.to_csv(p_clusters, index=False)
    with open(done_flag, 'w', encoding='utf-8') as f_out:
        f_out.write('OK\n')

    return {
        'folds': df_folds,
        'summary': df_summary,
        'horizons': df_horizons,
        'clusters': df_clusters,
    }


def compare_m1_common_rows(
    base_slug: str,
    variant_slugs: List[str],
    work_dir: str,
    out_csv: Optional[str] = None,
) -> pd.DataFrame:
    """Doi chieu cong bang M1 tren tap dong chung voi Baseline."""
    p_base_folds = os.path.join(work_dir, f'{base_slug}_folds.csv')
    df_base_f = pd.read_csv(p_base_folds).set_index(['Fold', 'Model'])
    p_dir = os.path.join(work_dir, 'preds')
    rows: List[Dict[str, Any]] = []

    for v_slug in variant_slugs:
        df_var_f = pd.read_csv(os.path.join(work_dir, f'{v_slug}_folds.csv'))
        for r in df_var_f.itertuples(index=False):
            ctrl = df_base_f.loc[(r.Fold, r.Model)]
            p_b = os.path.join(
                p_dir, f'{base_slug}_{r.Fold}_{r.Model}.parquet'
            )
            p_v = os.path.join(
                p_dir, f'{v_slug}_{r.Fold}_{r.Model}.parquet'
            )
            if not (os.path.isfile(p_b) and os.path.isfile(p_v)):
                continue
            dfb = pd.read_parquet(p_b)
            dfv = pd.read_parquet(p_v).drop(columns=['y_true', 'trip_id'])
            pairs = dfb.merge(
                dfv,
                on='row_id',
                validate='one_to_one',
                suffixes=('_base', '_var'),
            )
            yt = pairs['y_true'].to_numpy(dtype=np.float64)
            eb = pairs['y_pred_base'].to_numpy(dtype=np.float64) - yt
            ev = pairs['y_pred_var'].to_numpy(dtype=np.float64) - yt
            mae_b = float(np.mean(np.abs(eb)))
            mae_v = float(np.mean(np.abs(ev)))
            rmse_b = float(np.sqrt(np.mean(eb ** 2)))
            rmse_v = float(np.sqrt(np.mean(ev ** 2)))
            le_m = yt <= 168.0
            gt_m = yt > 168.0
            d_le = (
                float(np.mean(np.abs(ev[le_m])) - np.mean(np.abs(eb[le_m])))
                if np.any(le_m)
                else np.nan
            )
            d_gt = (
                float(np.mean(np.abs(ev[gt_m])) - np.mean(np.abs(eb[gt_m])))
                if np.any(gt_m)
                else np.nan
            )
            rows.append({
                'Variant': r.Variant,
                'Fold': r.Fold,
                'Eval_Q': r.Eval_Q,
                'Model': r.Model,
                'Common_Rows': int(len(pairs)),
                'Common_Trips': int(pairs['trip_id'].nunique()),
                'Baseline_MAE_Common_hours': round(mae_b, 4),
                'Variant_MAE_Common_hours': round(mae_v, 4),
                'Delta_MAE_Common_hours': round(mae_v - mae_b, 4),
                'LE168_Delta_MAE_Common_hours': round(d_le, 4),
                'GT168_Delta_MAE_Common_hours': round(d_gt, 4),
                'Baseline_RMSE_Common_hours': round(rmse_b, 4),
                'Variant_RMSE_Common_hours': round(rmse_v, 4),
                'Delta_RMSE_Common_hours': round(rmse_v - rmse_b, 4),
                'Baseline_Row_Coverage_pct': float(
                    ctrl['Eval_Row_Coverage_pct']
                ),
                'Variant_Row_Coverage_pct': float(r.Eval_Row_Coverage_pct),
                'Baseline_Trip_Coverage_pct': float(
                    ctrl['Eval_Trip_Coverage_pct']
                ),
                'Variant_Trip_Coverage_pct': float(r.Eval_Trip_Coverage_pct),
                'Fewer_Predicted_Rows': int(ctrl['Eval_Rows'] - r.Eval_Rows),
                'Fewer_Predicted_Trips': int(
                    ctrl['Eval_Trips'] - r.Eval_Trips
                ),
            })

    df_cmp = pd.DataFrame(rows)
    out_p = out_csv or os.path.join(work_dir, 'm1_common_rows_comparison.csv')
    df_cmp.to_csv(out_p, index=False)
    return df_cmp


def package_results_zip(work_dir: str, zip_path: str) -> str:
    """Dong goi toan bo file ket qua CSV/MD/PNG vao 1 file ZIP."""
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(work_dir):
            dirs[:] = [d for d in dirs if d != 'preds']
            for fname in sorted(files):
                fpath = os.path.join(root, fname)
                if os.path.abspath(fpath) == os.path.abspath(zip_path):
                    continue
                arc = os.path.relpath(fpath, start=work_dir)
                zf.write(fpath, arcname=arc)
    return zip_path

# Dong bo lai tham so tu Cell 1 Config
if 'CFG_OVERRIDES' in globals():
    globals().update(CFG_OVERRIDES)
print(f'[ENGINE V2 READY] HAS_CUDA={HAS_CUDA} | 'f'MAX_TRIPS={MAX_TRIPS_PER_QUARTER} | RESUME={RESUME_FROM_DISK}')


In [ ]:
QUARTER_CACHE = load_quarter_cache(
    quarters=['Q1', 'Q2', 'Q3', 'Q4'],
    max_trips=MAX_TRIPS_PER_QUARTER,
    seed=SEED,
    recover_q1_raw_nans=False,
)
M5_VARIANTS = [
    ('B0_all20_raw', None, 'raw'),
    ('T1_all20_log1p', None, 'log1p'),
    ('S1_mi16_raw', 16, 'raw'),
    ('S2_mi12_raw', 12, 'raw'),
    ('ST1_mi16_log1p', 16, 'log1p'),
    ('ST2_mi12_log1p', 12, 'log1p'),
]
for v_name, k_f, t_mode in M5_VARIANTS:
    run_expanding_window_experiment(
        experiment_name='M5_Selection',
        variant_name=v_name,
        active_models=ACTIVE_MODELS,
        m5_cls=make_m5_class(k_f, t_mode, v_name),
        quarter_cache=QUARTER_CACHE,
        work_dir=WORK_DIR,
        resume_from_disk=RESUME_FROM_DISK,
        max_dl_windows=MAX_DL_WINDOWS_PER_SPLIT,
    )

In [ ]:
"""Tong hop Master Summary (All / LE_168h / GT_168h) & Dong goi ZIP."""

import glob

sum_files = sorted(glob.glob(os.path.join(WORK_DIR, '*_summary.csv')))
sum_files = [
    p for p in sum_files
    if not os.path.basename(p).startswith(('master_', 'runner'))
]
if sum_files:
    df_master = pd.concat(
        [pd.read_csv(p) for p in sum_files], ignore_index=True
    )
    p_master = os.path.join(
        WORK_DIR, 'm5_v2_master_summary.csv'
    )
    df_master.to_csv(p_master, index=False)
    cols_view = [
        c for c in ['Experiment', 'Variant', 'Model', 'Num_Features',
                    'MAE_hours', 'LE168_MAE_hours',
                    'GT168_MAE_hours', 'RMSE_hours', 'R2',
                    'MBE_bias_hours', 'Underprediction_pct',
                    'AsymCost_w2_5', 'PICP_90_pct']
        if c in df_master.columns
    ]
    print('=== M5_V2 MASTER SUMMARY ===')
    print(df_master[cols_view].to_string(index=False))

hor_files = sorted(glob.glob(os.path.join(WORK_DIR, '*_horizons.csv')))
hor_files = [
    p for p in hor_files
    if not os.path.basename(p).startswith(('master_', 'runner'))
]
if hor_files:
    df_h_all = pd.concat(
        [pd.read_csv(p) for p in hor_files], ignore_index=True
    )
    p_h = os.path.join(
        WORK_DIR, 'm5_v2_master_horizons.csv'
    )
    df_h_all.to_csv(p_h, index=False)

zip_out = ('/kaggle/working/m5_v2_results.zip' if IS_KAGGLE else os.path.join(WORK_DIR, 'm5_v2_results.zip'))
package_results_zip(WORK_DIR, zip_out)
print(f'[ZIP READY] {zip_out}')